https://quantum.cloud.ibm.com/computers?utm_source=chatgpt.com

# 7. Route C — Die Kontraktions-Eichung

## Worum es geht

Das Problem von Route B ist das der Companion-Operator $E$ die Gedächtnissumme nach $K$ Termen abschneidet. Alles, was länger als $K\Delta t$ zurückliegt, ist verloren (der Rest-Term $R_n$ aus der Tabelle in dem Abschnitt *Der Companionpropagator*) und die Dynamik wird ungenauer je weiter die Simulation fortschreitet. Route C löst das, idem es keinen Companiontensor, der nur für ein kleines Zeitfenster gilt und keine ADOs enthält, aus dem HEOM-Propagator berechnet, sondern den HEOM-Propagator, der die gesamte Dynamik enthält, selbst nimmt und ihn anstatt $E$ auf das Grid gibt:

$$P = \exp(\mathcal{L}_{\mathrm{HEOM}}\,\Delta t) \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}}\times\mathcal{D}_{\mathrm{tot}}},\qquad \mathcal{D}_{\mathrm{tot}} = \mathcal{N}_{\mathrm{ADOs}}\cdot d^2 .$$

Die Operatornorm von $E$ war mit $\sqrt{2}$ beschränkt. Die Operatornorm von $P$ ist jedoch leider viel größer: $\Vert P\Vert_2 = 32{,}39$. Nach Proposition 13 multiplizieren sich die Erfolgswahrscheinlichkeiten, und $32{,}39^{-200}$ ist ungefähr $10^{-301}$. Man kann jedoch eine geschickte Koordinatentransformation machen, sodass $\Vert P\Vert_2 \approx 1$. Danach kann man dann eine Arnoldicompression machen da $P$ wegen den ganzen ADOs ja eine sehr große Matrix ist (viel größer als $E$). Im Fall von $P$ kann man das $m$ jedoch so groß wählen wie man will ohne dass die Norm $\Vert P\Vert_21$ wieder zu wachsen beginnt.

## Scaled HEOM

Die normale HEOM formel lautet:
$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = \underbrace{ -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}}(t) }_{\text{Term 1: Eigendynamik \& Zerfall}} + \underbrace{ \sum_{j,k} \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t) }_{\text{Term 2: Kopplung nach Oben}} + \underbrace{ \sum_{j,k} n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t) }_{\text{Term 3: Kopplung nach Unten}}$$

Wir wollen das jetzt in scaled HEOM umwandeln, dass wie folgt aussieht:
$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = \underbrace{ -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}} }_{\text{Unskalierter Diagonalterm}} + \underbrace{ \sum_{j,k} \sqrt{(n_{jk}+1)\frac{|a_{jk}|}{\hbar}} \, \phi_j \hat{\sigma}_{\mathbf{n}_{jk,+}} }_{\text{Skaliert mit scaling\_up}} + \underbrace{ \sum_{j,k} \sqrt{\frac{n_{jk}}{|a_{jk}|/\hbar}} \, \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk,-}} }_{\text{Skaliert mit scaling\_down}}$$

Dazu definieren wir $$\tilde{\sigma}_{\mathbf{n}}(t) = S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}(t) \quad \text{mit} \quad S_{\mathbf{n}} = \sqrt{ \prod_{j,k} n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} }$$
und berechnen $$\frac{d}{dt} \Big(\hat{\sigma}_{\mathbf{n}} \Big) = \frac{1}{S_n}\frac{d}{dt} \Big( S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}} \Big) = \frac{1}{S_n} \text{Term 1}(S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}) + \frac{1}{S_n}\text{Term 2}(S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}}) + \frac{1}{S_n} \text{Term 3}(S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}})$$

### Term 1
$$ \frac{1}{S_n} \text{Term 1}(S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}})= \frac{1}{S_n} \left( -\frac{i}{\hbar}[H, S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}] + \sum_{j,k} \left( \tilde{\phi}_j\tilde{\theta}_{j,0} - n_{jk}\gamma_{jk} \right) S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}} \right) = \frac{1}{S_n} S_{\mathbf{n}} \left( -\frac{i}{\hbar}[H, \hat{\sigma}_{\mathbf{n}}] + \sum_{j,k} \left( \tilde{\phi}_j\tilde{\theta}_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}} \right) = \text{Term 1}(\hat{\sigma}_{\mathbf{n}})$$

### Term 2
$$\begin{align*}
\frac{1}{S_{\mathbf{n}}} \text{Term 2}\left(S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}}\right) &= \frac{1}{S_{\mathbf{n}}} \sum_{j,k} \tilde{\phi}_j \left( S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}} \right) = \sum_{j,k} \left( \frac{S_{\mathbf{n}_{jk+}}}{S_{\mathbf{n}}} \right) \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}} = \sum_{j,k} \sqrt{ \frac{ (n_{jk}+1)! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}+1} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} }{ n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} } } \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}} \\
&= \sum_{j,k} \sqrt{ \frac{ (n_{jk}+1) \cdot n_{jk}! \cdot \left( \frac{|a_{jk}|}{\hbar} \right) \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} }{ n_{jk}! \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} } } \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}} = \sum_{j,k} \sqrt{ (n_{jk}+1) \frac{|a_{jk}|}{\hbar} } \, \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}}
\end{align*}$$

### Term 3

$$\begin{align*}
\frac{1}{S_{\mathbf{n}}} \text{Term 3}\left(S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}}\right) &= \frac{1}{S_{\mathbf{n}}} \sum_{j,k} n_{jk} \tilde{\theta}_{j,k} \left( S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}} \right) = \sum_{j,k} n_{jk} \left( \frac{S_{\mathbf{n}_{jk-}}}{S_{\mathbf{n}}} \right) \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} n_{jk} \sqrt{ \frac{ (n_{jk}-1)! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} }{ n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} \\
&= \sum_{j,k} n_{jk} \sqrt{ \frac{ (n_{jk}-1)! \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} }{ n_{jk} \cdot (n_{jk}-1)! \cdot \left( \frac{|a_{jk}|}{\hbar} \right) \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} n_{jk} \sqrt{ \frac{ 1 }{ n_{jk} \left( \frac{|a_{jk}|}{\hbar} \right) } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} \sqrt{ n_{jk}^2 \cdot \frac{ 1 }{ n_{jk} \left( \frac{|a_{jk}|}{\hbar} \right) } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} \\
&= \sum_{j,k} \sqrt{ \frac{ n_{jk} }{ |a_{jk}| / \hbar } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}
\end{align*}$$

### Intuition hinter der Skalierung
Mit $\phi_j = i V_j^\times$ und $\theta_{j,k\neq0} = i a_{jk} V_j^\times$ wobei $a_{jk} = \frac{4 \lambda \gamma \nu_k}{\beta (\nu_k^2 - \gamma^2)}$ und
$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \big( \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t) + n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t) \big)$$
kann man sehen dass der Term $\phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t)$ keinen Vorfaktor enthält wohingegen der Term $n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t)$ den Forfaktor $(n_{jk}+1) \frac{a_{jk}}{\hbar}$ enthält. Dieses Ungleichgewicht der beiden Terme führt in der Matrix der HEOM gleichung (7.1) zu sehr verschiedenen Eigenwerten und erzeugt somit ein steifes Problem. Dieses lässt sich von numerischen Integratoren nur mit sehr kleinen Zeitschritten lösen und ist daher sehr teuer. Selbst wenn die Werte für $n_{jk}$ nur klein sind, pflanzen si sich durch die Hierarchie wie faktorielle fort was zu der großen Ungleichheit der Terme führt. Deshalb ist es besser ein scaling einzuführen um die Matrix zu Preconditionen und die eigenwerte besser aneinander anzupassen.

### 1. Transformation des HEOM-Generators

Der Gesamtzustandsvektor aller ADOs sei $\vec{\Sigma}(t)$, und die Dynamik ist gegeben durch:
$$\frac{\mathrm{d}}{\mathrm{d}t} \vec{\Sigma}(t) = \mathcal{L}_{\mathrm{HEOM}} \vec{\Sigma}(t) \tag{7.1}$$

Die umskalierte Koordinate lautet $\widetilde{\Sigma}_i(t) = \Sigma_i(t) / s_i$, wobei $s_i > 0$ die Einträge des Skalierungsvektors `scale` sind. In Matrixschreibweise mit der Diagonalmatrix $D = \operatorname{diag}(s_1, \dots, s_N)$ gilt:
$\widetilde{\vec{\Sigma}}(t) = D^{-1} \vec{\Sigma}(t)$ und deswegen $\vec{\Sigma}(t) = D\, \widetilde{\vec{\Sigma}}(t)$. Einsetzen in die Bewegungsgleichung (7.1) liefert:
$$\frac{\mathrm{d}}{\mathrm{d}t} \big(D\, \widetilde{\vec{\Sigma}}(t)\big) = \mathcal{L}_{\mathrm{HEOM}} \big(D\, \widetilde{\vec{\Sigma}}(t)\big) \qquad \implies \qquad\frac{\mathrm{d}}{\mathrm{d}t} \widetilde{\vec{\Sigma}}(t) = \underbrace{\big(D^{-1} \mathcal{L}_{\mathrm{HEOM}} D\big)}_{\widetilde{\mathcal{L}}_{\mathrm{HEOM}}} \widetilde{\vec{\Sigma}}(t)$$

Der transformierte Generator ist somit eine Ähnlichkeitstransformation: $\widetilde{\mathcal{L}}_{\mathrm{HEOM}} = D^{-1} \mathcal{L}_{\mathrm{HEOM}} D$. Da $D$ eine Diagonalmatrix ist, gilt $D_{jj} = s_j$ und $(D^{-1})_{ii} = \frac{1}{s_i}$. Für das Matrixelement $(i, j)$ des transformierten Operators folgt direkt:
$$\big(\widetilde{\mathcal{L}}_{\mathrm{HEOM}}\big)_{ij} = \big(D^{-1} \mathcal{L}_{\mathrm{HEOM}} D\big)_{ij} = (D^{-1})_{ii} \cdot (\mathcal{L}_{\mathrm{HEOM}})_{ij} \cdot D_{jj} = \frac{1}{s_i} \cdot (\mathcal{L}_{\mathrm{HEOM}})_{ij} \cdot s_j = (\mathcal{L}_{\mathrm{HEOM}})_{ij} \cdot \frac{s_j}{s_i}$$

Der Ausdruck `(scale[None, :] / scale[:, None])`ist ein äußeres Produkt und spannt eine Matrix der Dimension $(N \times N)$ mit genau den Einträgen $\frac{s_j}{s_i}$ an Position $(i, j)$ auf. Die wird dann elementweise auf $A$ multipliziert: `A = A * (scale[None, :] / scale[:, None])`

## Der HEOM Propagator $P$ bildet eine Halbgruppe

Im nicht-Markovschen Fall ist das Bad kein passiver Zuschauer: Es merkt sich Wechselwirkungen und tauscht Energie sowie Korrelationen mit dem System aus. Die Standard-HEOM (Hierarchical Equations of Motion) löst dieses Problem, indem sie den Zustandsraum erweitert. Wir fassen die vektorisierte Systemdichtematrix $\text{vec}(\rho_S)$ und das dynamische Gedächtnis des Bades in einem einzigen großen Super-Vektor $\vec{\Sigma}(t)$ zusammen:

$$\vec{\Sigma}(t) = \begin{pmatrix} \text{vec}(\rho_S(t)) \\ \text{vec}(\mathrm{ADO}_1(t)) \\ \vdots \\ \text{vec}(\mathrm{ADO}_{\mathcal{N}_{\mathrm{ADOs}}-1}(t)) \end{pmatrix} \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}}}$$

* **$\text{vec}(\rho_S)$ (oberster Block):** Die vektorisierte Systemdichtematrix der Dimension $d^2$ (*zeroth-order ADO*).
* **$\text{vec}(\mathrm{ADO}_k)$ (untere Blöcke):** Die vektorisierten Hilfsoperatoren (*Auxiliary Density Operators*), die jeweils ebenfalls $d \times d$-Matrizen darstellen und die quantenmechanischen Bad-Korrelationen speichern.
* **Gesamtdimension:** Da $\mathcal{N}_{\mathrm{ADOs}}$ Matrizen der Dimension $d \times d$ untereinander gestapelt sind, beträgt die Gesamtlänge des Vektors $\mathcal{D}_{\mathrm{tot}} = \mathcal{N}_{\mathrm{ADOs}} \cdot d^2$.

Auf diesem erweiterten Raum ist die Zeitevolution exakt und lokal in der Zeit (formal Markovsch) und gehorcht dem linearen Differentialgleichungssystem:

$$\frac{\mathrm{d}}{\mathrm{d}t} \vec{\Sigma}(t) = \mathcal{L}_{\mathrm{HEOM}} \vec{\Sigma}(t) \tag{7.1}$$

wobei der zeitunabhängige HEOM-Liouville-Superoperator $\mathcal{L}_{\mathrm{HEOM}}$ eine Matrix der Größe $\mathcal{D}_{\mathrm{tot}} \times \mathcal{D}_{\mathrm{tot}}$ darstellt. Die Zeitentwicklung über einen festen Zeitschritt $\Delta t$ lässt sich exakt über das Matrixexponential ausdrücken:

$$P = \exp(\mathcal{L}_{\mathrm{HEOM}} \Delta t) \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}} \times \mathcal{D}_{\mathrm{tot}}} \implies \vec{\Sigma}(t+\Delta t) = P \ @ \ \vec{\Sigma}(t)$$

Für eine Zeitentwicklung um $n$ Schritt muss man dann einfach den propagator $P$ $n$-mal auf den Startzustand bei $t=0$ anwenden:

$$\vec{\Sigma}(n\Delta t) = P^n\,\vec{\Sigma}(0),\qquad \vec{\Sigma}(0) = \begin{pmatrix}\mathrm{vec}(\rho_S(0))\\ \mathbf{0}\end{pmatrix} \tag{7.2}$$

> **Proposition 18.** Auf dem erweiterten Raum $\mathbb{C}^{\mathcal{D}_{\mathrm{tot}}}$ bildet die diskrete HEOM-Zeitentwicklung eine Halbgruppe: Der Zustand $\vec{\Sigma}_n = P\,\vec{\Sigma}_{n-1}$ hängt ausschließlich vom unmittelbaren Vorgänger ab.

**Beweis.** Da der HEOM-Liouvillian $\mathcal{L}_{\mathrm{HEOM}}$ zeitunabhängig ist, gilt für das Matrixexponential die Standard-Halbgruppeneigenschaft $\mathrm{e}^{\mathcal{L}(t+s)} = \mathrm{e}^{\mathcal{L}t}\,\mathrm{e}^{\mathcal{L}s}$ und somit $P(t+s) = P(t)P(s)$ bzw.$P^{n+m} = P^n P^m$. Für diskrete Zeitschritte folgt damit unmittelbar:

$$\vec{\Sigma}(n\Delta t) = \mathrm{e}^{\mathcal{L}n\Delta t}\vec{\Sigma}(0) = \mathrm{e}^{\mathcal{L}\Delta t}\,\mathrm{e}^{\mathcal{L}(n-1)\Delta t}\vec{\Sigma}(0) = \mathrm{e}^{\mathcal{L}\Delta t}\, \vec{\Sigma}\big((n-1)\Delta t\big) = P\,\vec{\Sigma}\big((n-1)\Delta t\big). \quad \blacksquare$$

Das ist eine reine Ein-Schritt-Rekursion: Zur Bestimmung von $\vec{\Sigma}_n$ genügt allein der Zustand $\vec{\Sigma}_{n-1}$ aus dem letzten Schritt – frühere Zeitschritte werden nicht benötigt. Im gegensatz zu $E$ entsteht somit auf dem Gesamtraum (System + Bad) per Konstruktion kein Gedächtnis-Rest ($R_n = 0$).

## Das neue Hindernis: $\Vert P\Vert_2 = 32{,}39$

Proposition 13 aus Kapitel 5 lässt sich direkt auf diesen Fall übertragen: Wir müssen lediglich $E$ durch den Propagator $P$ und $X_0$ durch den erweiterten Vektor $\vec{\Sigma}_0$ ersetzen. Da der Beweis rein auf dem Teleskopprodukt basiert und keine speziellen Eigenschaften von $E$ voraussetzt, gilt für die Gesamterfolgswahrscheinlichkeit $p_{\mathrm{total}}$ hier mit $\vec{\Sigma}_0$ und $P$ exakt dieselbe Zerlegung:

$$p_{\mathrm{total}} = \underbrace{\frac{1}{s^{2n}}}_{\text{Teil 1 (Gatter-Strafe)}}\cdot\underbrace{\frac{\Vert P^n\vec{\Sigma}_0\Vert^2}{\Vert\vec{\Sigma}_0\Vert^2}}_{\text{Teil 2 (physikalische Längenänderung)}},\qquad s = \Vert P\Vert_2 \tag{7.3}$$

Während bei Route B der Teil 2 harmlos war und Teil 1 das Problem darstellte, offenbart sich hier ein noch lehrreicheres Phänomen: **Beide Faktoren nehmen absurde Werte an** — und zwar in entgegengesetzte Richtungen.

**Teil 1: Die Gatter-Strafe.** Für unser konkretes Modell liefert die numerische Auswertung leider einen sehr kleinen Wert:

$$\Vert P\Vert_2 = 32{,}39 \quad\implies\quad \text{Teil 1} = \frac{1}{32{,}39^{200}} \approx 8{,}17\times10^{-303}$$

**Teil 2: Die Längenänderung.** Physikalisch würde man erwarten, dass dissipative Prozesse die Zustände dämpfen und dieser Faktor $\le 1$ bleibt. Entlang der tatsächlichen Trajektorie *wächst* die standardmäßige euklidische Länge des Vektors jedoch drastisch an, was unsere Erfolgswahrscheinlichkeit erhöht:

| $n$ | $0$ | $1$ | $2$ | $5$ | $10$ | $25$ | $50$ | $100$ |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| $\Vert P^n\vec\Sigma_0\Vert/\Vert\vec\Sigma_0\Vert$ | $1$ | $1{,}98$ | $19{,}8$ | $432$ | $1872$ | $2950$ | $2717$ | $1101$ |

Das Maximum der relativen Norm erreicht sogar einen Wert von $3165$. Für $n = 100$ Zeitschritte ergibt sich damit $\text{Teil 2} = 1101^2 \approx 1{,}21\times10^{6}$. 

Kombiniert man beide Terme, reproduziert dies exakt die gemessene winzige Erfolgswahrscheinlichkeit:

$$p_{\mathrm{total}}(100) = \underbrace{8{,}17\times10^{-303}}_{\text{Teil 1}}\cdot\underbrace{1{,}21\times10^{6}}_{\text{Teil 2}} \approx 9{,}90\times10^{-297}$$

## 7.3 Die Eichung: ein Skalarprodukt, in dem $\mathcal{L}$ dissipativ ist

das Ziel ist es eine Ähnlichkeitstransfomation $P := W^{1/2}\,P\,W^{-1/2}$ zufinden die $||P||_2 \approx 1$ erfüllt. Das Werkzeug dafür ist die Lyapunov-Gleichung. Im Folgenden zeigen wir zuerst, dass 
$$W := \int_0^\infty \Big(\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\Big)^\dagger\,\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\,\mathrm d\tau = \int_0^\infty \mathrm{e}^{-2\delta\tau}\,\mathrm{e}^{\mathcal{L}^\dagger\tau}\,\mathrm{e}^{\mathcal{L}\tau}\,\mathrm d\tau$$
alle gewünschten Eigenschaften (Hermitizität und positive Definitheit) erfüllt und eine valide Lösung der Lyapunov-Gleichung ist. Da $W$ hermitesch ($W = W^\dagger$) und strikt positiv definit ist, garantiert der Spektralsatz eine unitäre Diagonalisierung der Form $W = V D V^\dagger = \sum_{i} w_i v_i v_i^\dagger$ mit orthonormalen Eigenvektoren $V$ ($V^\dagger V = \mathbb{I}$) und strikt positiven reellen Eigenwerten $w_i > 0$. Über den Funktionalkalkül definieren wir die eindeutige positive Matrixwurzel $W^{1/2}$ sowie deren Inverse $W^{-1/2}$ direkt über die Spektraldarstellung:

$$W^{1/2} := V D^{1/2} V^\dagger = V \begin{pmatrix} \sqrt{w_1} & & 0 \\ & \ddots & \\ 0 & & \sqrt{w_n} \end{pmatrix} V^\dagger$$

$$W^{-1/2} := (W^{1/2})^{-1} = V D^{-1/2} V^\dagger = V \begin{pmatrix} \frac{1}{\sqrt{w_1}} & & 0 \\ & \ddots & \\ 0 & & \frac{1}{\sqrt{w_n}} \end{pmatrix} V^\dagger$$

Diese Matrizen bilden die gesuchte Gauge-Transformation $\widetilde{P} = W^{1/2} P W^{-1/2}$, welche die Normkontraktion im transformierten Koordinatensystem erzwingt.

Anschließend zeigen wir, dass die Lyapunov-Gleichung unter diesen Bedingungen eindeutig lösbar ist und $W$ somit die einzige Lösung darstellt. Schlussendlich nutzen wir die Lyapunov-Gleichung, um eine Schranke für die Norm von $P$ herzuleiten. 

Dabei verwenden wir den spektralen Shift $\delta > 0$, um den Realteil der Eigenwerte von $\mathcal{L}$ strikt in die linke Halbebene zu verschieben: Sei $v$ ein Eigenvektor von $\mathcal{L}$ zum Eigenwert $\lambda$, also gilt per Definition $\mathcal{L} v = \lambda v$. Wenden wir nun die verschobene Matrix $(\mathcal{L} - \delta\mathbb{I})$ auf denselben Vektor $v$ an:

$$(\mathcal{L} - \delta\mathbb{I}) v = \mathcal{L} v - \delta\mathbb{I} v = \lambda v - \delta v = (\lambda - \delta) v$$

Das zeigt, dass der Vektor $v$ exakt derselbe Eigenvektor bleibt und der neue zugehörige Eigenwert zwingend $\lambda - \delta$ lautet. Da dies für jeden beliebigen Eigenvektor $v_i$ von $\mathcal{L}$ gilt, verschiebt sich ausnahmslos jeder Eigenwert $\lambda_i \to \lambda_i - \delta$.

> **Proposition 20 (Lyapunov-Eichung).** Sei $\delta>0$ so gewählt, dass $\mathcal{L}-\delta\mathbb{I}$ Hurwitz ist, d.h. $\mathrm{Re}\,\lambda < \delta$ für jeden Eigenwert $\lambda$ von $\mathcal{L}$. Dann besitzt die Lyapunov-Gleichung
> $$(\mathcal{L}-\delta\mathbb{I})^\dagger W + W(\mathcal{L}-\delta\mathbb{I}) = -\,\mathbb{I} \tag{7.5}$$
> genau eine Lösung $W$; diese ist hermitesch und positiv definit, und mit $\Vert x\Vert_W := \Vert W^{1/2}x\Vert_2$ gilt
> $$\Vert e^{\mathcal{L}\tau}\Vert_W \; := \; \Vert W^{1/2}e^{\mathcal{L}\tau}\Vert_2 \;\le\; e^{\delta\tau}\qquad\text{für alle }\tau\ge0 . \tag{7.6}$$

**Beweis.** Der Beweis gliedert sich in vier Schritte:

**(a) Wohldefiniertheit und Positivität des Lösungsansatzes:**  
Da $\delta\mathbb{I}$ mit $\mathcal{L}$ kommutiert, faktorisiert das Matrixexponential gemäß $\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau} = \mathrm{e}^{-\delta\tau}\mathrm{e}^{\mathcal{L}\tau}$. Wir definieren den Lösungsansatz (Kandidaten) für $W$ als Integral über alle zukünftigen Zustände der verschobenen Dynamik:

$$W := \int_0^\infty \Big(\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\Big)^\dagger\,\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\,\mathrm d\tau = \int_0^\infty \mathrm{e}^{-2\delta\tau}\,\mathrm{e}^{\mathcal{L}^\dagger\tau}\,\mathrm{e}^{\mathcal{L}\tau}\,\mathrm d\tau \tag{7.7}$$

Weil der Generator $\mathcal{L}-\delta\mathbb{I}$ Hurwitz ist (alle Eigenwert-Realteile sind echt negativ), zerfällt der Propagator $\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}$ für $\tau \to \infty$ exponentiell. Wenn die Matrix diagonalisierbar ist kann man das schön sehen:
$$A = V D V^{-1} = V \begin{pmatrix} \lambda_1 & 0 & \cdots \\ 0 & \lambda_2 & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} V^{-1} \implies \mathrm{e}^{A\tau} = V \mathrm{e}^{D\tau} V^{-1} = V \begin{pmatrix} \mathrm{e}^{\lambda_1 \tau} & 0 & \cdots \\ 0 & \mathrm{e}^{\lambda_2 \tau} & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} V^{-1} \xrightarrow{\tau \to \infty} \begin{pmatrix} 0 & 0 & \cdots \\ 0 & 0 & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} = \mathbf{0}$$
Es gilt jedoch auch wenn die Matrix nicht diagonalisierbar ist. Somit existieren Konstanten $c,\eta>0$ mit $\Vert \mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\Vert_2 \le c\,\mathrm{e}^{-\eta\tau}$. Der Integrand erhält in der 2-Norm durch $c^2\mathrm{e}^{-2\eta\tau}$ also eine obere Schranke, womit das Integral garantiert nicht divergiert und $W$ daher wohlbestimmt ist.

* **Hermitezität ($W = W^\dagger$):** Der Integrand hat für jedes $\tau$ die Gestalt $Z^\dagger Z$ und ist damit symmetrisch/hermitesch.  
* **Strikte positive Definitheit ($x^\dagger W x > 0$):** Für jeden Testvektor $x \neq 0$ summiert das Integral die Längenquadrate der Trajektorie auf:
  $$x^\dagger W x = \int_0^\infty \big\Vert \mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}x\big\Vert_2^2\,\mathrm d\tau > 0.$$
  Da der Integrand stetig und nichtnegativ ist und beim Startwert $\tau=0$ den strikt positiven Wert $\Vert x\Vert_2^2 > 0$ annimmt, ist die Gesamtsumme $x^\dagger W x$ echt größer als null. Die Bedingung $x^\dagger W x > 0$ ist gleichbedeutend damit, dass alle Eigenwerte echt positiv sind. Somit existiert die Wurzel $\sqrt{\lambda_i}$ im Reellen, und die Transformationsmatrix $W^{1/2} = V\,\mathrm{diag}(\sqrt{\lambda_i})\,V^\dagger$ ist vollständig invertierbar ohne Division durch null.

**(b) Der Ansatz löst die Lyapunov-Gleichung (7.5):**  
Wir betrachten den Integranden $G(\tau) := \big(\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\big)^\dagger \mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}$ und bilden dessen Zeitableitung. Mit der Produktregel folgt unmittelbar:

$$\frac{\mathrm dG}{\mathrm d\tau} = (\mathcal{L}-\delta\mathbb{I})^\dagger G(\tau) + G(\tau)(\mathcal{L}-\delta\mathbb{I}).$$

Integrieren wir diese Beziehung auf beiden Seiten von $\tau = 0$ bis $\tau = \infty$, so liefert die linke Seite nach dem Hauptsatz der Differential- und Integralrechnung einen einfachen Randterm:

$$\int_0^\infty \frac{\mathrm dG}{\mathrm d\tau}\,\mathrm d\tau = \Big[G(\tau)\Big]_0^\infty = G(\infty) - G(0) = \mathbf{0} - \mathbb{I} = -\mathbb{I},$$

da $G(\infty) = \mathbf{0}$ (exponentieller Zerfall aus Schritt a) und $G(0) = \mathrm{e}^{\mathbf{0}} = \mathbb{I}$ gilt. Auf der rechten Seite ziehen wir die konstanten Generatormatrizen vor das Integral:

$$-\mathbb{I} = (\mathcal{L}-\delta\mathbb{I})^\dagger \underbrace{\int_0^\infty G(\tau)\,\mathrm d\tau}_{=\,W} + \underbrace{\int_0^\infty G(\tau)\,\mathrm d\tau}_{=\,W} (\mathcal{L}-\delta\mathbb{I}) \quad\implies\quad (\mathcal{L}-\delta\mathbb{I})^\dagger W + W(\mathcal{L}-\delta\mathbb{I}) = -\mathbb{I}.$$

Damit erfüllt $W$ exakt die kontinuierliche Lyapunov-Gleichung.

**(c) Eindeutigkeit der Lösung:**  
Um zu zeigen, dass es genau eine Matrix $W$ gibt, die die Lyapunov-Gleichung löst, fassen wir die linke Seite von (7.5) als lineare Abbildung auf dem Raum aller Matrizen auf (einen sogenannten *Sylvester-Operator*):

$$\mathcal{S}(X) := (\mathcal{L}-\delta\mathbb{I})^\dagger X + X(\mathcal{L}-\delta\mathbb{I})$$

Die Gleichung lautet damit kompakt $\mathcal{S}(W) = -\mathbb{I}$. Aus der linearen Algebra ist bekannt: Ein lineares Gleichungssystem besitzt genau dann für jede rechte Seite eine eindeutige Lösung, wenn der zugehörige Operator **keinen Eigenwert gleich null** hat (also invertierbar ist).

* **Woher kommen die Eigenwerte von $\mathcal{S}$?**  
  Der Operator $\mathcal{S}(X) = A^\dagger X + X A$ wirkt auf eine Matrix $X$ gleichzeitig von links (über $A^\dagger$) und von rechts (über $A$). 
  
  Wählt man als Testmatrix das äußere Produkt $X = u v^\dagger$ aus einem Eigenvektor $u$ von $A^\dagger$ (mit Eigenwert $\overline{\lambda_j}$, also $A^\dagger u = \overline{\lambda_j} u$) und einem Eigenvektor $v$ von $A$ (mit $A v = \lambda_i v \implies v^\dagger A = \lambda_i v^\dagger$), sieht man die Wirkung direkt:
  
  $$\begin{aligned}
  \mathcal{S}(u v^\dagger) 
  &= \big(A^\dagger u\big) v^\dagger + u \big(v^\dagger A\big) = \big(\overline{\lambda_j} u\big) v^\dagger + u \big(\lambda_i v^\dagger\big) = (\overline{\lambda_j} + \lambda_i)\,(u v^\dagger)
  \end{aligned}$$
  
  Die Matrix $X = u v^\dagger$ verhält sich also exakt wie ein Eigenvektor des Operators $\mathcal{S}$, und der zugehörige Eigenwert ist schlicht die Summe $\overline{\lambda_j} + \lambda_i$.
* **Warum kann kein Eigenwert null sein?**  
  Da $(\mathcal{L}-\delta\mathbb{I})$ Hurwitz ist, besitzen ausnahmslos alle Eigenwerte einen strikt negativen Realteil: $\mathrm{Re}(\lambda_k) < 0$. Für jede beliebige Summe zweier Eigenwerte gilt daher:
  $$\mathrm{Re}(\overline{\lambda_j} + \lambda_i) = \underbrace{\mathrm{Re}(\lambda_j)}_{<\,0} + \underbrace{\mathrm{Re}(\lambda_i)}_{<\,0} < 0 \quad\implies\quad \overline{\lambda_j} + \lambda_i \neq 0.$$

Da kein einziger Eigenwert von $\mathcal{S}$ auf der Null liegt, ist der Kern trivial ($\ker(\mathcal{S}) = \{\mathbf{0}\}$). Der Operator $\mathcal{S}$ ist bijektiv invertierbar, und die Lösung $W = \mathcal{S}^{-1}(-\mathbb{I})$ existiert und ist strikt eindeutig.

**(d) Herleitung der Normabschätzung (7.6):**  
In diesem letzten Schritt zeigen wir, wie die Wahl von $W$ garantiert, dass die Dynamik in der neuen Metrik nicht unkontrolliert anwachsen kann.

* **Die Idee (Lyapunov-Funktion als „Energie“):**  
  Wir betrachten das Längenquadrat des Zustandsvektors in der $W$-Metrik als eine verallgemeinerte Energie:
  $$V(\tau) := \Vert x(\tau)\Vert_W^2 = \big\Vert W^{1/2}x(\tau)\big\Vert_2^2 = \big(W^{1/2}x(\tau)\big)^\dagger \big(W^{1/2}x(\tau)\big) = x(\tau)^\dagger W x(\tau)$$
  Ziel ist es zu prüfen, wie sich diese Energie entlang einer echten physikalischen Trajektorie $\dot{x} = \mathcal{L}x$ über die Zeit verändert.

* **Schritt 1: Zeitableitung von $V(\tau)$ bilden**  
  Mit der Produktregel für Skalarprodukte und dem Einsetzen der Bewegungsgleichung $\dot{x} = \mathcal{L}x$ (bzw. $\dot{x}^\dagger = x^\dagger \mathcal{L}^\dagger$) erhalten wir:
  $$\frac{\mathrm dV}{\mathrm d\tau} = \dot{x}^\dagger W x + x^\dagger W \dot{x} = x^\dagger \big(\mathcal{L}^\dagger W + W\mathcal{L}\big) x$$

* **Schritt 2: Die Lyapunov-Gleichung einsetzen**  
  Wir stellen die Lyapunov-Gleichung $(\mathcal{L}-\delta\mathbb{I})^\dagger W + W(\mathcal{L}-\delta\mathbb{I}) = -\mathbb{I}$ nach dem Term $\mathcal{L}^\dagger W + W\mathcal{L}$ um:
  $$\mathcal{L}^\dagger W + W\mathcal{L} = -\mathbb{I} + 2\delta W$$
  Eingesetzt in die Zeitableitung ergibt sich:
  $$\frac{\mathrm dV}{\mathrm d\tau} = x^\dagger \big(-\mathbb{I} + 2\delta W\big) x = \underbrace{-x^\dagger x}_{=\,-\Vert x\Vert_2^2} + 2\delta \underbrace{x^\dagger W x}_{=\,V(\tau)}$$

* **Schritt 3: Der dissipative Kern**  
  Der Term $-\Vert x\Vert_2^2$ ist für jeden Zustand $x \neq 0$ **strikt negativ**. Er wirkt wie eine permanente Reibung, die dem System Energie entzieht. Lässt man diesen negativen Verlustterm weg, erhält man eine obere Schranke:
  $$\frac{\mathrm dV}{\mathrm d\tau} = -\Vert x\Vert_2^2 + 2\delta V(\tau) \;\le\; 2\delta V(\tau) \iff \frac{\mathrm dV}{\mathrm d\tau} - 2\delta V(\tau) \le 0$$

* **Schritt 4: Integration über den Zeitverlauf**  
  Multiplikation mit dem strikt positiven integrierenden Faktor $\mathrm{e}^{-2\delta\tau} > 0$ ergibt
  $\mathrm{e}^{-2\delta\tau} \frac{\mathrm dV}{\mathrm d\tau} - 2\delta \mathrm{e}^{-2\delta\tau} V(\tau) \le 0$. Durch Anwendung der Produktregel in umgekehrter Richtung erhalten wir:
  $$\frac{\mathrm d}{\mathrm d\tau} \Big(\mathrm{e}^{-2\delta\tau} V(\tau)\Big) \le 0$$
  
  Integration beider Seiten bezüglich der Zeit von $0$ bis $\tau$:
  $\int_0^\tau \frac{\mathrm d}{\mathrm ds} \Big(\mathrm{e}^{-2\delta s} V(s)\Big)\,\mathrm ds \le \int_0^\tau 0\,\mathrm ds$ und auswertung über den Hauptsatz der Differential- und Integralrechnung:
  $$\Big[\mathrm{e}^{-2\delta s} V(s)\Big]_0^\tau \le 0 \implies \mathrm{e}^{-2\delta\tau} V(\tau) - \underbrace{\mathrm{e}^{0}}_{=\,1} V(0) \le 0$$
  
  Umstellen und Multiplikation mit $\mathrm{e}^{2\delta\tau} > 0$:
  $$\mathrm{e}^{-2\delta\tau} V(\tau) \le V(0) \implies V(\tau) \le \mathrm{e}^{2\delta\tau} V(0)$$

* **Schritt 5: Rückübersetzung auf die Operatornorm**  
  Ersetzen wir $V(\tau) = \Vert x(\tau)\Vert_W^2$ und $V(0) = \Vert x_0\Vert_W^2$ und ziehen die Quadratwurzel, erhalten wir mit $x(\tau) = \mathrm{e}^{\mathcal{L}\tau} x_0$:
  $$\Vert x(\tau)\Vert_W \le \mathrm{e}^{\delta\tau} \Vert x_0\Vert_W \iff \frac{\big\Vert \mathrm{e}^{\mathcal{L}\tau} x_0\big\Vert_W}{\Vert x_0\Vert_W} \le \mathrm{e}^{\delta\tau}$$

  Da $x(\tau) = \mathrm{e}^{\mathcal{L}\tau} x_0$ für jeden beliebigen Startzustand $x_0$ gilt, folgt über das Supremum aller normierten Startvektoren unmittelbar die Schranke für die induzierte Operatornorm:
  $$\Vert \mathrm{e}^{\mathcal{L}\tau}\Vert_W = \sup_{x_0 \neq 0} \frac{\Vert \mathrm{e}^{\mathcal{L}\tau} x_0\Vert_W}{\Vert x_0\Vert_W} \le \mathrm{e}^{\delta\tau}. \qquad \blacksquare$$

### Was die Eichung mit dem Propagator macht

> **Proposition 21.** Mit $W$ aus Proposition 20 sei
> $$\tilde P := W^{1/2}\,P\,W^{-1/2},\qquad \tilde{\vec\Sigma}_0 := W^{1/2}\vec{\Sigma}_0 .$$
> Dann gilt
> $$\Vert\tilde P\Vert_2 \le e^{\delta\Delta\tau} \qquad\text{und}\qquad \tilde P^{\,n} = W^{1/2}P^{n}W^{-1/2} , \tag{7.8}$$
> insbesondere ist die Trajektorie $\tilde{\vec\Sigma}_n = \tilde P^{\,n}\tilde{\vec\Sigma}_0$ **exakt** und man liest daraus
> $$\mathrm{vec}\,\rho_S(n\Delta t) = Q\,W^{-1/2}\,\tilde{\vec\Sigma}_n \tag{7.9}$$
> ab. Die Eichung führt also **keinerlei** Näherung ein.

**Beweis.** Da die Wurzel $W^{1/2}$ invertierbar ist, können wir jeden Vektor $y$ eindeutig als $y = W^{1/2}x$ schreiben. Für die Norm substituieren wir daher im Supremum $y = W^{1/2}x$. Weil $W^{1/2}$ invertierbar ist, ist das eine Bijektion, das Supremum läuft also über dieselbe Menge:

$$\begin{aligned}
\Vert\tilde P\Vert_2 = \sup_{y\neq0}\frac{\Vert \tilde{P}y\Vert_2}{\Vert y\Vert_2} = \sup_{y\neq0}\frac{\Vert W^{1/2}PW^{-1/2}y\Vert_2}{\Vert y\Vert_2}
\;\overset{y=W^{1/2}x}{=}\; \sup_{x\neq0}\frac{\Vert W^{1/2}Px\Vert_2}{\Vert W^{1/2}x\Vert_2}
= \sup_{x\neq0}\frac{\Vert Px\Vert_W}{\Vert x\Vert_W}
= \Vert P\Vert_W = \Vert e^{\mathcal{L}\Delta\tau}\Vert_W \;\overset{(7.6)}{\le}\; e^{\delta\Delta\tau}
\end{aligned}$$

Die Potenz ist ein Teleskopprodukt, in dem sich $W^{-1/2}W^{1/2}=\mathbb{1}$ immer wieder aufhebt:

$$\tilde P^{\,n} = \big(W^{1/2}PW^{-1/2}\big)^n = W^{1/2}P\,\underbrace{W^{-1/2}W^{1/2}}_{=\,\mathbb{1}}\,P\,\underbrace{W^{-1/2}W^{1/2}}_{=\,\mathbb{1}}\,P\cdots P\,W^{-1/2} = W^{1/2}P^{n}W^{-1/2}$$

 Startet man im transformierten Bild bei $\tilde{\vec{\Sigma}}_0 = W^{1/2}\vec{\Sigma}_0$, lautet der Zustand nach $n$ Schritten:

  $$\tilde{\vec{\Sigma}}_n = \tilde{P}^n \tilde{\vec{\Sigma}}_0 = \big(W^{1/2}P^n W^{-1/2}\big)\big(W^{1/2}\vec{\Sigma}_0\big) = W^{1/2}\big(P^n\vec{\Sigma}_0\big) = W^{1/2}\vec{\Sigma}_n.$$

  Multipliziert man von links mit $W^{-1/2}$, erhält man den exakten physikalischen Zustand $\vec{\Sigma}_n = W^{-1/2}\tilde{\vec{\Sigma}}_n$ zurück. Die abschließende Projektion $Q$ liefert genau die gesuchte System-Dichtematrix $\rho_S(n\Delta\tau)$ gemäß (7.9). $\;\blacksquare$

Jetzt lesen wir Gleichung (7.3) noch einmal, aber in der neuen Eichung:
$$
\begin{aligned}
p_{\mathrm{total}} 
&= \underbrace{\frac{1}{\Vert\tilde{P}\Vert_2^{2n}}}_{\text{Teil 1 (Gatter-Strafe)}} \cdot \underbrace{\frac{\big\Vert\tilde{P}^n \tilde{\vec{\Sigma}}_0\big\Vert_2^2}{\Vert\tilde{\vec{\Sigma}}_0\Vert_2^2}}_{\text{Teil 2 (Längenänderung)}}
= \frac{1}{\Vert W^{1/2} P W^{-1/2}\Vert_2^{2n}} \cdot \frac{\big\Vert \overbrace{(W^{1/2} P W^{-1/2}) \cdots (W^{1/2} P W^{-1/2})}^{n\text{-mal}} (W^{1/2} \vec{\Sigma}_0)\big\Vert_2^2}{\Vert W^{1/2} \vec{\Sigma}_0\Vert_2^2} \\
&= \frac{1}{\Vert P\Vert_W^{2n}} \cdot \frac{\big\Vert W^{1/2} P^n \underbrace{W^{-1/2} W^{1/2}}_{=\,\mathbb{I}} \vec{\Sigma}_0\big\Vert_2^2}{\Vert W^{1/2} \vec{\Sigma}_0\Vert_2^2} = \frac{1}{\Vert P\Vert_W^{2n}} \cdot \frac{\big\Vert W^{1/2} (P^n \vec{\Sigma}_0)\big\Vert_2^2}{\Vert W^{1/2} \vec{\Sigma}_0\Vert_2^2} 
= \underbrace{\frac{1}{\Vert P\Vert_W^{2n}}}_{\text{Teil 1}} \cdot \underbrace{\frac{\Vert P^n \vec{\Sigma}_0\Vert_W^2}{\Vert\vec{\Sigma}_0\Vert_W^2}}_{\text{Teil 2}}
\end{aligned}
$$

- Teil 1: Das Gatter wird aus $\tilde P$ gebaut, der Skalierungsfaktor ist also $s = \Vert\tilde P\Vert_2 \le e^{\delta\Delta\tau}$, und die Gatter-Strafe über $n$ Schritte wird zu $$s^{2n} \;\le\; \Big(e^{\delta\Delta\tau}\Big)^{2n} = e^{2\delta\,n\Delta\tau} = e^{2\delta T} \tag{7.10}$$ Man wählt $\delta$ so klein, dass $\delta T \ll 1$ ist, und Teil 1 aus (7.3) ist erledigt. Da $T \approx 10^{-12}$ kann $\delta$ auch sehr groß gewählt werden.
- Teil 2 wird jetzt ebenfalls in der $W$-Norm gemessen, und dort ist es durch $e^{2\delta T}$ nach oben beschränkt: Nach Proposition 20 gilt für jeden Zeitschritt $\tau = n\Delta t = T$ und für jeden beliebigen Startvektor: $\Vert \mathrm{e}^{\mathcal{L}T} x_0\Vert_W \le \mathrm{e}^{\delta T} \Vert x_0\Vert_W$. Da $P^n \vec{\Sigma}_0 = \mathrm{e}^{\mathcal{L}n\Delta t} \vec{\Sigma}_0 = \mathrm{e}^{\mathcal{L}T}\vec{\Sigma}_0$ ist, setzen wir $x_0 = \vec{\Sigma}_0$ ein:$$\Vert P^n \vec{\Sigma}_0\Vert_W \le \mathrm{e}^{\delta T} \Vert \vec{\Sigma}_0\Vert_W$$Teilt man durch $\Vert \vec{\Sigma}_0\Vert_W$ und quadriert beide Seiten, erhält man direkt die obere Schranke für Teil 2:$$\text{Teil 2} = \frac{\Vert P^n \vec{\Sigma}_0\Vert_W^2}{\Vert \vec{\Sigma}_0\Vert_W^2} = \frac{\Vert W^{1/2} P^n \vec{\Sigma}_0\Vert_2^2}{\Vert  W^{1/2} \vec{\Sigma}_0\Vert_2^2} = \frac{\Vert\tilde{P}^n \tilde{\vec{\Sigma}}_0\Vert_2^2}{\Vert\tilde{\vec{\Sigma}}_0\Vert_2^2} \;\le\; \Big(\mathrm{e}^{\delta T}\Big)^2 = \mathrm{e}^{2\delta T}$$

Die Strafe hängt also **nur noch vom Produkt $\delta T$** ab, nicht mehr von der Zahl der Schritte (aber vom Zeithorizont $T$) und nicht von der Nicht-Normalität von $P$. 

**Warum darf $\delta$ nicht null sein?** Dass $\delta > 0$ gewählt werden muss, ist eine direkte physikalische Konsequenz des thermischen Gleichgewichts:
* Der Liouvillian $\mathcal{L}$ besitzt den Eigenwert $\lambda_0 = 0$, da der stationäre Gleichgewichtszustand $\vec{\Sigma}_{\mathrm{ss}}$ zeitunabhängig ist ($\mathcal{L}\vec{\Sigma}_{\mathrm{ss}} = 0 = 0 \cdot \vec{\Sigma}_{\mathrm{ss}}$).
* Ohne Shift ($\delta = 0$) divergiert die quadratische Form von $W$ entlang dieses stationären Zustands:
  $$
  \begin{aligned}
  \vec{\Sigma}_{\mathrm{ss}}^\dagger W \vec{\Sigma}_{\mathrm{ss}} 
  &= \int_0^\infty \vec{\Sigma}_{\mathrm{ss}}^\dagger \mathrm{e}^{\mathcal{L}^\dagger\tau} \mathrm{e}^{\mathcal{L}\tau} \vec{\Sigma}_{\mathrm{ss}}\,\mathrm d\tau 
  = \int_0^\infty \big\Vert \underbrace{\mathrm{e}^{\mathcal{L}\tau} \vec{\Sigma}_{\mathrm{ss}}}_{=\,\vec{\Sigma}_{\mathrm{ss}}}\big\Vert_2^2\,\mathrm d\tau = \int_0^\infty \Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2\,\mathrm d\tau 
  = \Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2 \cdot \int_0^\infty 1\,\mathrm d\tau 
  = \Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2 \cdot [\tau]_0^\infty 
  = \infty
  \end{aligned}
  $$
* Mit dem Regularisierungsfaktor $\delta > 0$ wird der Integrand exponentiell gedämpft, wodurch das Integral wohlbestimmt konvergiert:
  $$\vec{\Sigma}_{\mathrm{ss}}^\dagger W \vec{\Sigma}_{\mathrm{ss}} = \int_0^\infty \mathrm{e}^{-2\delta\tau} \Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2\,\mathrm d\tau = \Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2 \left[ -\frac{\mathrm{e}^{-2\delta\tau}}{2\delta} \right]_0^\infty = \frac{\Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2}{2\delta} < \infty$$
* Die Lyapunov-Gleichung ist ohne Shift unlösbar: Für $\delta = 0$ lautet die Lyapunov-Gleichung $\mathcal{L}^\dagger W + W\mathcal{L} = -\mathbb{I}$. Multipliziert man diese von links und rechts mit $\vec{\Sigma}_{\mathrm{ss}}^\dagger$ bzw. $\vec{\Sigma}_{\mathrm{ss}}$, erhält man wegen $\mathcal{L}\vec{\Sigma}_{\mathrm{ss}} = 0$:$$\underbrace{\vec{\Sigma}_{\mathrm{ss}}^\dagger \mathcal{L}^\dagger W \vec{\Sigma}_{\mathrm{ss}}}_{=\,0} + \underbrace{\vec{\Sigma}_{\mathrm{ss}}^\dagger W \mathcal{L} \vec{\Sigma}_{\mathrm{ss}}}_{=\,0} = -\vec{\Sigma}_{\mathrm{ss}}^\dagger \mathbb{I} \vec{\Sigma}_{\mathrm{ss}} \implies 0 = -\Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2 < 0$$Das ist ein direkter Widerspruch ($0 = -\Vert\vec{\Sigma}_{\mathrm{ss}}\Vert_2^2$). Die Lyapunov-Gleichung besitzt für $\delta = 0$ schlicht keine Lösung.



Gemessen für unser Modell ($T = 1000\,\mathrm{fs}$, also $T = 0{,}1884\,\mathrm{cm}$): bei $\delta = 0{,}1$ fällt $\Vert\tilde P^{\,n}\tilde{\vec\Sigma}_0\Vert/\Vert\tilde{\vec\Sigma}_0\Vert$ über 100 Schritte auf $0{,}455$, Teil 2 ist also $0{,}21$. Genau das ist der Punkt der Eichung: Sie macht **beide** Faktoren einzeln zu Zahlen der Ordnung $1$, statt zwei astronomische Zahlen gegeneinander wegkürzen zu lassen.

| $\delta$ [cm$^{-1}$] | $\Vert\tilde P\Vert_2$ | Schranke $e^{\delta\Delta\tau}$ | $\mathrm{cond}(W)$ | $p_{\mathrm{total}}(100)$ |
| ---: | ---: | ---: | ---: | ---: |
| $0{,}02$ | $1{,}000038$ | $1{,}000038$ | $1{,}74\times10^{10}$ | $0{,}365$ |
| $0{,}1$ | $1{,}000188$ | $1{,}000188$ | $1{,}68\times10^{10}$ | $0{,}200$ |
| $0{,}5$ | $1{,}000942$ | $1{,}000942$ | $1{,}62\times10^{10}$ | $0{,}129$ |
| $2{,}0$ | $1{,}003774$ | $1{,}003774$ | $1{,}42\times10^{10}$ | $0{,}062$ |
| $10{,}0$ | $1{,}019015$ | $1{,}019015$ | $7{,}48\times10^{9}$ | $0{,}0037$ |

Zwei Dinge fallen auf. Erstens ist die Schranke (7.6) **scharf** — die gemessene Norm trifft $e^{\delta\Delta\tau}$ auf sechs Stellen. Zweitens ändert sich $\mathrm{cond}(W)$ über einen Faktor $500$ in $\delta$ hinweg kaum; sie wird also nicht vom Shift bestimmt, sondern von der Spannweite der ADO-Skalen im HEOM-Ansatz. $\delta$ ist damit ein freier Parameter, den man für lange Läufe einfach kleiner macht.

Zum Vergleich derselbe Propagator, ungeeicht und geeicht:

$$\Vert P\Vert_2 = 32{,}39 \quad\longrightarrow\quad \Vert\tilde P\Vert_2 = 1{,}000038 \qquad\text{und}\qquad p_{\mathrm{total}}(100):\quad 10^{-301}\;\longrightarrow\;0{,}365$$

## 7.4 Jetzt wird die Arnoldi-Kompression gratis

$\tilde P$ operiert noch immer auf $\mathbb{C}^{2640}$ ($\lceil\log_2 2640\rceil + 1 = 13$ Qubits). Wir komprimieren daher wie zuvor via Arnoldi-Verfahren, nun jedoch auf dem geeichten Propagator: Mit dem Krylov-Raum $\tilde{\mathcal{K}}_m := \mathcal{K}_m(\tilde P, \tilde{\vec\Sigma}_0)$, der Orthonormalbasis $\tilde Q_m$ und der komprimierten Matrix $\tilde H_m := \tilde Q_m^\dagger \tilde P \tilde Q_m$ gilt:

> **Proposition 22.** Für **jedes** $m$ gilt: $\Vert \tilde H_m\Vert_2 \le \Vert\tilde P\Vert_2 \le \mathrm{e}^{\delta\Delta\tau} \approx 1$.

**Beweis.** Da $\tilde Q_m$ eine Isometrie ist, folgt $\Vert \tilde H_m\Vert_2 \le \Vert \tilde P\Vert_2$ direkt aus den Eigenschaften der orthogonalen Projektion. Die zweite Schranke liefert Proposition 21. $\;\blacksquare$

Damit entfällt der fundamentale Trade-off bisheriger Routen:
* **Bisher (z. B. Route B):** Ein zu großes $m$ nahm unphysikalische Richtungen auf; $\Vert H_m\Vert_2$ stieg gegen die nutzlose Schranke $\sqrt{2}$, was die Erfolgswahrscheinlichkeit zerstörte (z. B. $p \sim 10^{-18}$ bei $m=64$).
* **Mit Kontraktions-Eichung:** Weil die obere Schranke nun $\Vert\tilde H_m\Vert_2 \le 1{,}000038$ lautet, bleibt $\tilde H_m$ für **beliebiges** $m$ eine Kontraktion. 

Die Krylov-Dimension $m$ ist somit kein heikler Kompromissparameter mehr, sondern ein **reiner Konvergenzparameter**: Man kann $m$ bedenkenlos so lange erhöhen, bis die gewünschte Genauigkeit erreicht ist. $H_m$ sit dasnn eine $m \times m$ Matrix, was $\lceil\log_2 m\rceil$ Qubits auf dem circuit kostet.

| $m$ | Qubits | $\Vert\tilde H_m\Vert_2$ | Abweichung von QuTiP | $p_{\mathrm{total}}(100)$ |
| ---: | ---: | ---: | ---: | ---: |
| $16$ | $5$ | $1{,}000038$ | $1{,}93\times10^{-1}$ | $0{,}172$ |
| $32$ | $6$ | $1{,}000038$ | $6{,}22\times10^{-2}$ | $0{,}332$ |
| $64$ | $7$ | $1{,}000038$ | $\mathbf{5{,}27\times10^{-7}}$ | $0{,}365$ |
| $96$ | $8$ | $1{,}000038$ | $5{,}27\times10^{-7}$ | $0{,}365$ |
| $128$ | $8$ | $1{,}000038$ | $5{,}27\times10^{-7}$ | $0{,}365$ |

Zwei Aspekte sind hierbei zentral:

Erstens stagniert die Abweichung zu QuTiP ab $m = 64$ bei einem Plateau von $5{,}27 \times 10^{-7}$.  Genau diese Eigenschaft fehlte ohne Eichung: dort wäre $p_{\mathrm{total}}$ ab einem gewissen $m$ wieder eingebrochen. Dies stellt keinen Approximationsfehler der Arnoldi-Kompression dar, sondern markiert die numerische Integrationstoleranz des QuTiP-Referenzsolvers.

Um die tatsächliche Konvergenz der Arnoldi-Kompression ohne Solver-Artefakte offenzulegen, vergleicht man die komprimierte Dynamik stattdessen direkt mit der exakten, unkomprimierten Referenztrajektorie. Betrachtet man einen langen Zeithorizont von $n = 500$ Zeitschritten (bei $\delta = 0{,}1\,\mathrm{cm}^{-1}$), fällt dieser reine Kompressionsfehler streng monoton ab:
* $m = 32$: $3{,}15 \times 10^{-1}$
* $m = 64$: $2{,}91 \times 10^{-2}$
* $m = 96$: $1{,}97 \times 10^{-4}$
* $m = 128$: $1{,}65 \times 10^{-8}$
* $m = 192$: $3{,}35 \times 10^{-14}$

Es tritt keinerlei Instabilität oder Fehlerumkehr bei großen Krylov-Dimensionen auf.

**Registergröße.** Von $\mathcal{D}_{\mathrm{tot}} = 2640$ (12 System-Qubits) schrumpfen wir auf $m = 128$ (7 System-Qubits), zusammen mit der Ancilla also **8 Qubits**. Route B kam mit $m=32$ auf 6 Qubits, ist also etwas sparsamer — dafür aber um vier Größenordnungen ungenauer und braucht 25 klassisch gerechnete Zeitschritte. Bei $m=64$ liegt Route C mit 7 Qubits schon bei $5{,}27\times10^{-7}$.

## 7.5 Das Gitter und die Ablesung

Die $m\times m$-Matrix $\tilde H_m$ wird auf $m_p = 2^{\lceil\log_2 m\rceil}$ aufgefüllt und **einmal** nach Kapitel 5 dilatiert:

$$U = \begin{pmatrix} \tilde H_m/s & B\\ C & -(\tilde H_m/s)^\dagger\end{pmatrix},\qquad B = \sqrt{\mathbb{1}-\tfrac{\tilde H_m\tilde H_m^\dagger}{s^2}},\qquad C = \sqrt{\mathbb{1}-\tfrac{\tilde H_m^\dagger\tilde H_m}{s^2}},\qquad s = \Vert\tilde H_m\Vert_2$$

Danach besteht der gesamte Schaltkreis aus genau einem Gatter, das man wiederholt:

```
krylov (7 Qubits) : ────[ U ]─────────────[ U ]─────────────[ U ]────────── ... ───> Y_n
                         │                 │                 │
Ancilla           : |0>──[ U ]─(M)──|0>────[ U ]─(M)──|0>────[ U ]─(M)─|0>─ ...
                               │                 │                 │
klass. Register   : ───────────●─────────────────●─────────────────●─────── ...
                             rec[1]            rec[2]            rec[3]
```

Zwei Unterschiede zum Bild in dem Abschnitt *Die Umsetzung auf dem Quantencomputer* fallen auf.

**Erstens brauchen wir keine Zustandspräparation.** Der Startvektor des Krylov-Raums ist per Konstruktion $q_1 = \tilde{\vec\Sigma}_0/\Vert\tilde{\vec\Sigma}_0\Vert$, in Krylov-Koordinaten also $y_0 = \Vert\tilde{\vec\Sigma}_0\Vert\,e_1$. Das Register startet damit ohnehin schon im richtigen Zustand $\vert 0\cdots0\rangle$; die gesamte Information über $\rho_S(0)$ und die Eichung steckt in der klassischen Zahl $\Vert\tilde{\vec\Sigma}_0\Vert$ und in der Basis $\tilde Q_m$.

**Zweitens messen wir die Ancilla sofort nach jedem Schritt und setzen sie zurück**, statt für jeden Schritt ein frisches Ancilla-Qubit zu spendieren. Nach dem Prinzip der aufgeschobenen Messung, das wir im Abschnitt *Many steps* diskutiert haben, ist das exakt äquivalent: Sobald Schritt $t$ vorbei ist, berührt kein Gatter dieses Ancilla-Qubit je wieder, die Messung kommutiert also mit allem Folgenden. Der Vorteil ist, dass wir mit **einem** Ancilla-Qubit für beliebig viele Schritte auskommen, statt $n$ davon zu brauchen. Das Messprotokoll wandert stattdessen in das klassische Register `rec`.

### Der akzeptierte Zweig ist deterministisch

> **Proposition 24.** Bedingt auf das Messprotokoll $\mathtt{rec} = 0\cdots0$ ist der Zustand des Krylov-Registers nach $t$ Schritten
> $$\vert Y_t\rangle = \frac{\tilde H_m^{\,t}\,y_0}{\Vert\tilde H_m^{\,t}\,y_0\Vert} \tag{7.12}$$
> und damit **unabhängig von jedem Zufall in den Messungen**.

**Beweis.** Induktion über $t$. Für $t=0$ ist nichts zu zeigen. Sei das Register vor Schritt $t$ im reinen Produktzustand $\vert Y_{t-1}\rangle\otimes\vert0\rangle_E$. Die Anwendung von $U$ spaltet ihn nach dem Abschnitt *One step* in genau zwei Zweige auf:

$$U\big(\vert0\rangle_E\otimes\vert Y_{t-1}\rangle\big) = \vert0\rangle_E\otimes\Big(\tfrac{\tilde H_m}{s}Y_{t-1}\Big) \;+\; \vert1\rangle_E\otimes\big(C\,Y_{t-1}\big)$$

Die Messung wählt zufällig einen der beiden Zweige aus — welchen, das ist die einzige Zufälligkeit im ganzen Ablauf. *Gegeben* das Ergebnis $\vert0\rangle$ projiziert der Kollaps aber auf den ersten Summanden, und die Normierung, die die Natur automatisch vornimmt, hebt den Faktor $1/s$ wieder auf:

$$\vert Y_t\rangle = \frac{\big(\tilde H_m/s\big)Y_{t-1}}{\big\Vert\big(\tilde H_m/s\big)Y_{t-1}\big\Vert} = \frac{\tfrac1s\,\tilde H_mY_{t-1}}{\tfrac1s\,\Vert\tilde H_mY_{t-1}\Vert} = \frac{\tilde H_m\,Y_{t-1}}{\Vert\tilde H_m\,Y_{t-1}\Vert}$$

Das ist eine **Funktion von $Y_{t-1}$ allein**, ohne jeden Rest an Zufall. Das anschließende `reset` bringt die Ancilla wieder nach $\vert0\rangle$, so dass die Induktionsvoraussetzung für Schritt $t+1$ wiederhergestellt ist. Setzt man die Rekursion $t$-mal ein und benutzt $y_0 = \Vert\tilde{\vec\Sigma}_0\Vert\,e_1 \propto Y_0$, folgt (7.12). $\;\blacksquare$

Das hat eine sehr praktische Konsequenz. Der Zufall im Schaltkreis steckt *ausschließlich* in der Frage, **ob** ein Durchlauf akzeptiert wird, nicht darin, **was** man im Erfolgsfall vorfindet. Ein einziger akzeptierter Shot trägt daher bereits den exakten Zustand; die Statistik über viele Shots wird nur für die skalare Zahl $p_{\mathrm{total}}$ gebraucht.

### Die Rückrechnung auf die Physik

Der Quantencomputer gibt uns den normierten Vektor $Y_t$. Um daraus $\rho_S(t)$ zu machen, brauchen wir zwei Dinge: seine Länge und die Rücktransformation aus den Krylov- und Eichkoordinaten. Die Länge liefert Proposition 13, wörtlich übertragen mit $E\to\tilde H_m$:

$$p_{\mathrm{total}}(t) = \frac{\Vert\tilde H_m^{\,t}y_0\Vert^2}{s^{2t}\Vert y_0\Vert^2} \quad\Longrightarrow\quad \Vert y_t\Vert \;=\; \underbrace{\Vert\tilde{\vec\Sigma}_0\Vert}_{=\,\Vert y_0\Vert}\cdot\, s^{\,t}\,\sqrt{p_{\mathrm{total}}(t)} \tag{7.13}$$

Die Rücktransformation fassen wir in einer einzigen klassischen Matrix zusammen, die **einmal** vorab berechnet wird:

$$R := \Big(W^{-1/2}\,\tilde Q_m\Big)_{[\,1:d^2,\;:\,]} \;\in\;\mathbb{C}^{d^2\times m} \qquad\Longrightarrow\qquad \mathrm{vec}\,\rho_S(t\Delta t) = R\;y_t \tag{7.14}$$

Denn nach (7.9) ist $\mathrm{vec}\rho_S = \Pi \, W^{-1/2}\tilde{\vec\Sigma}_t$, nach der Kompression ist $\tilde{\vec\Sigma}_t \approx \tilde Q_m y_t$, und $\Pi$ schneidet genau die ersten $d^2$ Zeilen heraus. Zusammengesetzt:

$$\mathrm{vec}\,\rho_S(t\Delta t) = \Pi \underbrace{\,W^{-1/2}\tilde Q_m}_{=\;R}\;\underbrace{\Vert\tilde{\vec\Sigma}_0\Vert\,s^{\,t}\sqrt{p_{\mathrm{total}}(t)}}_{=\;\lambda_t}\;\cdot\;Y_t$$

| Größe | Bedeutung | Woher kommt der Wert? | Wann im Ablauf? |
| :--- | :--- | :--- | :--- |
| $\Vert\tilde{\vec\Sigma}_0\Vert$ | Länge des geeichten Startvektors | $\Vert W^{1/2}\vec\Sigma_0\Vert$ | **vor** dem Quanten-Run, rein klassisch |
| $s$ | Dilatations-Skalierung | $s = \Vert\tilde H_m\Vert_2 \le e^{\delta\Delta\tau}$ | **vor** dem Quanten-Run, rein klassisch |
| $R$ | Rücktransformation in den Systemblock | $R = (W^{-1/2}\tilde Q_m)_{[1:d^2,:]}$ | **vor** dem Quanten-Run, rein klassisch |
| $p_{\mathrm{total}}(t)$ | kumulierte Erfolgswahrscheinlichkeit | Anteil der Shots mit $\mathtt{rec}[1..t] = 0\cdots0$ | **nach** dem Run, aus dem klassischen Register |
| $Y_t$ | normierter Registerzustand | Zustand der $\lceil\log_2 m\rceil$ Krylov-Qubits, bedingt auf die akzeptierten Shots | am Zeitschritt $t$ |

**Eine zweite, unabhängige Ablesung.** Es gibt einen Weg, der ohne $p_{\mathrm{total}}$ auskommt und somit das statistische Rauschen (Shot Noise) der Erfolgszählung eliminiert, was das ergebnis genauer macht. Die HEOM-Dynamik erhält die Spur des nullten ADOs, es gilt also zu jeder Zeit exakt $\mathrm{Tr}\,\rho_S(t) = 1$. Das ist keine Zusatzinformation über die Lösung, sondern eine Eigenschaft der Bewegungsgleichung, und sie legt den fehlenden Skalar fest:

$$\rho_S(t) = \frac{\mathrm{unvec}\big(R\,Y_t\big)}{\mathrm{Tr}\,\mathrm{unvec}\big(R\,Y_t\big)}$$

**Beweis:** Der unnormierte Vektor $y_t$ und der vom Quantencomputer ausgegebene, normierte Zustand $Y_t$ unterscheiden sich nur durch einen reellen, positiven Skalierungsfaktor $\lambda_t > 0$:$$y_t = \lambda_t \cdot Y_t \qquad \text{mit } \lambda_t = \Vert\tilde{\vec{\Sigma}}_0\Vert \cdot s^t \sqrt{p_{\mathrm{total}}(t)}$$Wendet man die lineare Rücktransformationsmatrix $R$ an, gilt wegen der Linearität:$$\operatorname{vec}(\rho_S(t)) = R \, y_t = R \, (\lambda_t Y_t) = \lambda_t \cdot (R Y_t)$$Wenden wir nun die lineare Operation $\operatorname{unvec}$ (das Umformen des $d^2$-Vektors in eine $d \times d$-Matrix) und anschließend die Spur $\operatorname{Tr}$ an:$$\operatorname{Tr}\big(\rho_S(t)\big) = \operatorname{Tr}\Big(\operatorname{unvec}\big(\lambda_t \cdot R Y_t\big)\Big) = \lambda_t \cdot \operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big)$$Da die physikalische Dichtematrix zu jedem Zeitpunkt strikt spurerhaltend ist ($\operatorname{Tr}(\rho_S(t)) = 1$), folgt zwingend:$$1 = \lambda_t \cdot \operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big) \quad\implies\quad \lambda_t = \frac{1}{\operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big)}$$ Setzt man diesen Skalar $\lambda_t$ nun oben wieder ein, bekommt man: $$\rho_S(t) = \lambda_t \cdot \operatorname{unvec}(R Y_t) = \left(\frac{1}{\operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big)}\right) \cdot \operatorname{unvec}(R Y_t) \, \qquad \blacksquare$$

Nebenbei erledigt das auch die globale Phase, die ein normierter Quantenzustand ohnehin nur bis auf $e^{i\varphi}$ festlegt: Die physikalische Spur muss reell und positiv sein. Beide Ablesungen benutzen **disjunkte** Information — die eine das Messprotokoll, die andere die Spurerhaltung —, ihre Übereinstimmung ist also ein in sich geschlossener Test des ganzen Kapitels.

## Zusammenfassung des Kapitels

Der Weg in einem Absatz: Die HEOM-Hierarchie ist auf dem erweiterten Raum bereits eine Halbgruppe, $\vec\Sigma_n = P^n\vec\Sigma_0$ ist exakt (Proposition 18), und der ganze Gedächtnis-Rest $R_n$ aus Route B entsteht erst dadurch, dass man mit $Q$ auf den Systemblock projiziert und die ADOs wegwirft. Wer sie behält, braucht kein $K$ und keine klassisch vorberechnete Historie. Das einzige Hindernis ist dann, dass $P$ mit $\rho(P)=1$ zwar brav, mit $\Vert P\Vert_2 = 32{,}39$ aber stark nicht-normal ist, und dass die Sz.-Nagy-Dilatation stur die Operatornorm berechnet — ein Hindernis, das auch feinere Zeitschritte nicht beseitigen (Proposition 19). Die Auflösung ist, dass $32{,}39$ eine Aussage über das *Skalarprodukt* ist: Die Lyapunov-Gleichung (7.5) liefert eine Metrik $W$, in der $P$ eine Kontraktion bis auf $e^{\delta\Delta\tau}$ ist (Proposition 20), ohne die Trajektorie im Geringsten zu verändern (Proposition 21). In dieser Eichung wird die Arnoldi-Kompression zum reinen Konvergenzparameter (Propositionen 22 und 23), und der akzeptierte Zweig des Schaltkreises ist deterministisch (Proposition 24).

$$\boxed{\;\Vert P\Vert_2 = 32{,}39 \;\xrightarrow{\;W^{1/2}\cdot W^{-1/2}\;}\; \Vert\tilde P\Vert_2 = 1{,}000038 \qquad\Longrightarrow\qquad p_{\mathrm{total}}(100):\;\;10^{-301}\;\longrightarrow\;0{,}37\;}$$

| Proposition | Aussage |
| :---: | :--- |
| **18** | HEOM ist auf dem ADO-Raum eine Halbgruppe; $R_n \equiv 0$ ohne jedes $K$ |
| **20** | Lyapunov-Eichung: $\exists\,W\succ0$ mit $\Vert e^{\mathcal{L}\tau}\Vert_W \le e^{\delta\tau}$ |
| **21** | $\tilde P = W^{1/2}PW^{-1/2}$ ist ähnlich zu $P$ — die Eichung ist exakt |
| **22** | $\Vert\tilde H_m\Vert_2 \le e^{\delta\Delta\tau}$ für jedes $m$ — Kompression bleibt Kontraktion |
| **24** | der auf $0\cdots0$ bedingte Zweig ist deterministisch — ein Shot genügt für den Zustand |

Die Implementierung liegt in `heom_gauge.py` (Erzeuger, Eichung, Arnoldi, Dilatation) und `gauge_circuit.py` (das Gitter); die Rechnungen und alle Abbildungen stehen in `main_durchgaengiges_grid.ipynb`.

# Messergebnisse der Diagonalterme in die richtige Basis transformieren

In der Theorie verwenden wir die Gleichung $$\mathrm{vec}\,\rho_S(t\Delta t) = \Pi \underbrace{\,W^{-1/2}\tilde Q_m}_{=\;R}\;\underbrace{\Vert\tilde{\vec\Sigma}_0\Vert\,s^{\,t}\sqrt{p_{\mathrm{total}}(t)}}_{=\;\lambda_t}\;\cdot\;Y_t$$ um den Zustand $Y_t$ am Ende des Quantencircuits mit der Matrix $R$ wieder in die richtige Basis zu transformieren um die Systemdichtematrix zu erhalten. Wir hatten ja zuvor eine Basistransformation (Ähnlichkeitstransformation) mit $W$ durchgeführt. In einer realen Anwenfung können wir den Zustand $Y_t$ aber nicht so einfach messen, dafür wäre Qunatentomographie nötig. Wir können laos nur counts auslesen. Angenommen, ein System aus $n$ Qubits befindet sich vor der Messung in einer quantenmechanischen Überlagerung $\vert{}\psi\rangle = c_0 \vert{}00\dots0\rangle + c_1 \vert{}00\dots1\rangle + \dots + c_{2^n-1} \vert{}11\dots1\rangle = \sum_{k=0}^{2^n-1} c_k \vert{}k\rangle$. Führt man einen einzelnen Shot aus:
- Das Quantensystem kollabiert instantan auf genau einen Basiszustand $\vert{}k\rangle$.
- Die Hardware gibt einen klassischen Bitstring aus (z. B. '010').
- Man erhält nur ein einziges Bitmuster, keine Information über die Amplituden $c_k$.

Um die Amplituden zu rekonstruieren, wiederholt man das gesamte Experiment (Präparation $\to$ Gatter $\to$ Messung) $N_{\text{shots}}$ Mal (z. B. $N = 10\,000$). Dabei führt man eine Strichliste und zählt, wie oft jeder Bitstring $k$ gemessen wurde. Nach dem Gesetz der großen Zahlen nähert sich die relative Häufigkeit für große Shot-Zahlen exakt der quantenmechanischen Wahrscheinlichkeit an:$$P(k) = \lim_{N_{\text{shots}} \to \infty} \frac{N_k}{N_{\text{shots}}} = \vert{}c_k\vert{}^2$$

Wir erhlaten also nur wahrscheinlichkeiten und keinen vollen Vektor aus dem Messprozess und können die Transformation mittels $R$ auf das ursprüngliche Koordinatensystem nicht mehr durchführen. Um das Problem zu lösen, transformieren wir den Zustand noch auf dem Quantencircuit in die richtige Basis. Dazu wenden wir ganz am ende ein zusätzliche Gate $B_j$ an. Um herzuleiten wie das gate $B_j$ auszusehen hat, machen wir folgende folgendes Beispiel.

Wir wollen die Population $\rho_{jj}$ der $j$-ten Site berechnen. Dazu greifen wir die entsprechende Zeile $r_j$ aus der Rücktransformationsmatrix $R$ heraus. Unser Zielwert ist das einfache Skalarprodukt:$$\rho_{jj} = r_j \cdot y_t = \sum_{k=1}^m r_{jk} y_{t,k}$$
Damit wir den Vektor $r_j$ in der Quantenwelt nutzen können, machen wir ihn zu einem gültigen, auf Länge $1$ normierten Quantenzustand: $\vert{}\chi_j\rangle := \frac{r_j^\dagger}{\Vert{}r_j\Vert{}_2}$. Wenn wir nun das quantenmechanische Skalarprodukt zwischen diesem neuen Zustand $\vert{}\chi_j\rangle$ und unserem Registerzustand $\vert{}y_t\rangle$ bilden, erhalten wir exakt unseren Zielwert, nur skaliert:$$\langle \chi_j \vert{} y_t \rangle = \frac{r_j}{\Vert{}r_j\Vert{}_2} \cdot y_t = \frac{\rho_{jj}}{\Vert{}r_j\Vert{}_2} \quad \implies \quad \rho_{jj} = \Vert{}r_j\Vert{}_2 \cdot \langle \chi_j \vert{} y_t \rangle$$ Wir müssten nun messen, "wie viel $\vert{}\chi_j\rangle$" im Zustand $\vert{}y_t\rangle$ steckt. Ein Quantencomputer kann jedoch nur messen wie viel eines Zustandes der Rechenbasis $\mathcal{B}_{\text{comp}} = \{\vert{}00\dots0\rangle, \vert{}00\dots1\rangle, \dots, \vert{}11\dots1\rangle\}$ im Endzustand des cicuits steckt. Also bauen wir uns einen Adapter: Ein unitäres Gatter $B_j$, das den Basis-Zustand $\vert{}0\dots0\rangle$ exakt auf unseren Wunschzustand $\vert{}\chi_j\rangle$ dreht:

$$ B_j \vert{}0\dots0\rangle = \vert{}\chi_j\rangle \qquad \implies \qquad \langle\chi_j|= \langle 0\dots0| B_j^\dagger $$

Gemessen werden dann nach der bornschen Regel natürlich nur das Betragsquadrat $\big\vert{} \langle 0\dots0 \vert{} B_j^\dagger \vert{} y_t \rangle \big\vert{}^2 \equiv q_j$. Fügen wir auch noch die Skalierungen $\Vert{}\widetilde{\vec{\Sigma}}_0\Vert{}_2$ und $s^t$ aus der Normierung des Eingagsvektors und des Propagators $\tilde{P}$ sowie die Erfolgswahrscheinlichkeit $\sqrt{p_{\text{total}}(t)}$ im richtigen Systemzustand zu landen hinzu, erhlaten wir schlussendlich:
$$\rho_{jj}(t) = \underbrace{\Vert{}r_j\Vert{}_2 \cdot \sqrt{q_j(t)}}_{\text{aus der Basis-Rotation}} \;\times\; \underbrace{\Vert{}\widetilde{\vec{\Sigma}}_0\Vert{}_2 \cdot s^t \cdot \sqrt{p_{\text{total}}(t)}}_{\text{Skalierung \& Überlebensrate}}$$

- $p_{\text{total}}(t)$ ist der Anteil aller Shots, die bei den Ancilla-Messungen durchgängig Nullen hatten.
- $q_j(t)$ ist der Anteil dieser überlebenden Shots, die nach der Rotation $B_j^\dagger$ im Systemregister lauter Nullen haben.


### Die Fehlerfortpflanzung und Zahl der benötigten shots
Das Ziel ist es vor dem Lauf auf einem Quantencomputer mit einer klassischen Rechnung die Zahl der nötigen Shots $N_{\text{shots, nötig}}$ zur Bestimmung der Populations mit einer bestimmten genauigkeit vorab zu berechnen. Da man hier die Dynamik klassisch lösen muss, steht dieser Weg einer echten Anwendung auf einem Quantencomputer nicht zur Verfügung.

Die rekonstruierte physikalische Population der Site $j$ zum Zeitpunkt $t$ lautet:$$\rho_{jj}(t) = \underbrace{\Vert{}r_j\Vert{}_2}_{\text{konstant}} \cdot \underbrace{\lambda_t}_{\text{konstant}} \cdot \sqrt{q_j(t) \cdot p_{\text{total}}(t)}$$Das Produkt $q_j \cdot p_{\text{total}}$ ist die absolute Wahrscheinlichkeit, dass ein einzelner Shot alle Ancilla-Messungen überlebt und am Ende exakt den Systemzustand $\vert{}0\dots0\rangle$ liefert. Die erwartete Anzahl an tatsächlichen Treffern (Counts) auf der Hardware ist somit:$$N_{\text{Treffer}} = q_j \cdot p_{\text{total}} \cdot N_{\text{shots}}$$
Die (meist sehr kleine) Gesamtwahrscheinlichkeit $P$, dass ein beliebiger Shot ein gültiger Treffer ist, beträgt somit:$$P = p_{\text{total}} \cdot q_j$$
- Erwartungswert (Mittelwert der Treffer): $\mu = N_{\text{shots}} \cdot P = N_{\text{shots}} \cdot p_{\text{total}} \cdot q_j = N_{\text{Treffer}}$
- Standardabweichung (mit Annahme $P \ll 1$): $\sigma = \sqrt{N_{\text{shots}} \cdot P \cdot (1 - P)} \approx \sqrt{N_{\text{shots}} \cdot P} = \sqrt{N_{\text{Treffer}}}$

Der relative Fehler einer reinen Zählrate (Verhältnis aus Standardabweichung und Mittelwert) ist daher der bekannte Poisson-Zählfehler:$$\frac{\sigma}{\mu} \approx \frac{\sqrt{N_{\text{Treffer}}}}{N_{\text{Treffer}}} = \frac{1}{\sqrt{N_{\text{Treffer}}}} = \frac{1}{2\sqrt{q_j \cdot p_{\text{total}} \cdot N_{\text{shots}}}}$$

Wenn wir beispielsweise eine relative Genauigkeit von $1\,\%$ ($\frac{\sigma}{\mu} = 10^{-2}$) fordern, können wir die Gleichung einfach nach der benötigten Shot-Zahl umstellen:$$N_{\text{shots, nötig}} \geq \frac{1}{4 \cdot 10^{-4} \cdot q_{\min} \cdot p_{t,\min}}$$

Um $q_j(t)$ und $p_{\text{total}}(t)$ vorab zu kennen, iteriert die Funktion den reduzierten Zustandsvektor klassisch durch die Zeit:$$y_{t} = H_m \cdot y_{t-1}$$Da der Krylov-Propagator $H_m$ nicht unitär ist (er enthält die Dissipation), schrumpft die Norm des Vektors physikalisch korrekterweise mit jedem Schritt.Die Überlebenswahrscheinlichkeit (Post-Selection) ist exakt der quadratische Normverlust relativ zur anvisierten Dilatations-Skalierung: $p_{\text{total}}(t) = \left( \frac{\Vert{}y_t\Vert{}}{s^t \Vert{}y_0\Vert{}} \right)^2$.Der Überlapp $q_j(t)$ ist schlicht das quantenmechanische Skalarprodukt des normierten Zustands $y_t / \Vert{}y_t\Vert{}$ mit unserem vorher definierten Zielzustand $\vert{}\chi_j\rangle$:$$q_j(t) = \big\vert{} \langle \chi_j \vert{} \frac{y_t}{\Vert{}y_t\Vert{}} \rangle \big\vert{}^2$$


### Herleitung von $B_j$
Das Ziel ist es, eine unitäre Matrix $B_j$ zu konstruieren, die genau die Bedingung $B_j \vert{}0\dots0\rangle = \vert{}\chi_j\rangle$ erfüllt. Der Zustand $\vert{}0\dots0\rangle$ ist in Vektorschreibweise einfach der erste Standardbasisvektor $e_1 = (1, 0, \dots, 0)^{\mathsf T}$. Wenn wir eine beliebige Matrix $B_j$ mit $e_1$ multiplizieren, greift diese Operation exakt die erste Spalte der Matrix heraus:$$B_j \vert{}0\dots0\rangle = \begin{pmatrix}  b_{11} & b_{12} & \dots & b_{1N} \\ b_{21} & b_{22} & \dots & b_{2N} \\ \vdots & \vdots & \ddots & \vdots \\ b_{N1} & b_{N2} & \dots & b_{NN} \end{pmatrix} \begin{pmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{pmatrix} = \begin{pmatrix} b_{11} \\ b_{21} \\ \vdots \\ b_{N1} \end{pmatrix} := (B_j)_{:, 1}$$
Unsere Aufgabe reduziert sich also auf eine rein geometrische Bedingung: Wir suchen eine beliebige unitäre Matrix, deren erste Spalte exakt unser Zielvektor $\vert{}\chi_j\rangle$ ist. Wie die restlichen Spalten aussehen, ist für unser Ergebnis völlig egal, solange sie alle senkrecht aufeinander und auf $\vert{}\chi_j\rangle$ stehen (damit die Matrix unitär bleibt). Das bedeutet, die Matrix muss diese Form haben:$$B_j = \Big( \vert{}\chi_j\rangle \quad\Big\vert{}\quad \vert{}v_2\rangle \quad\Big\vert{}\quad \dots \quad\Big\vert{}\quad \vert{}v_N\rangle \Big)$$ Um die Matrix $B$ unitär zu machen benutzen wie die QR-zerlegung. Damit die aus der QR-Zerlegung resultierende Matrix den vollen Raum aufspannt müssen die $|v_i\rangle$ linear unabhängig sein, sonst sind sie beliebig. Wir wählen $|v_i\rangle = |e_i\rangle$ und erhalten:
$$M = \Big( \vert{}\chi_j\rangle \quad\Big\vert{}\quad e_2 \quad\Big\vert{}\quad \dots \quad\Big\vert{}\quad e_N \Big) = \begin{pmatrix}  \chi_{j,1} & 0 & \dots & 0 \\ \chi_{j,2} & 1 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ \chi_{j,N} & 0 & \dots & 1 \end{pmatrix}$$
Diese Matrix $M$ hat jetzt zwar die richtige erste Spalte, aber sie ist nicht unitär, weil die Spalten $e_2 \dots e_N$ nicht senkrecht auf $\vert{}\chi_j\rangle$ stehen. Um die Spalten senkrecht aufeinander auszurichten, übergeben wir $M$ an den Algorithmus np.linalg.qr(M). Die QR-Zerlegung spaltet jede quadratische Matrix in das Produkt $M = Q \cdot R$:
- $Q$ ist eine perfekt unitäre Matrix (alle Spalten sind orthonormal).
- $R$ ist eine obere Dreiecksmatrix.

Weil $R$ unterhalb der Diagonale nur Nullen hat, entsteht die erste Spalte von $M$ ausschließlich aus der ersten Spalte von $Q$, multipliziert mit dem obersten linken Eintrag $R_{11}$:$$M_{:, 1} = Q_{:, 1} \cdot R_{11}$$Da wir wissen, dass $M_{:, 1} = \vert{}\chi_j\rangle$ bereits die Länge $1$ hat und die Spalte von $Q$ wegen der Unitarität ebenfalls Länge $1$ haben muss, folgt zwingend: $\vert{}R_{11}\vert{} = 1$.Der Eintrag $R_{11}$ ist also nur noch ein reiner Phasenfaktor $\mathrm{e}^{i\phi}$. Die erste Spalte von $Q$ ist somit bis auf diese Phase exakt unser gesuchter Vektor:$$Q_{:, 1} = \mathrm{e}^{-i\phi} \vert{}\chi_j\rangle$$
Die QR-Zerlegung hat uns eine gültige unitäre Matrix $Q$ geliefert, aber ihre erste Spalte hat vielleicht eine falsche Phase abbekommen. Der Algorithmus muss das noch korrigieren.Wir extrahieren die Diagonale von $R$ (die Vektorphasen) und teilen jeden Eintrag durch seinen eigenen Betrag, um reine Phasenfaktoren zu erhalten:$$\text{dg} = \frac{R_{kk}}{\vert{}R_{kk}\vert{}} = \mathrm{e}^{i\phi_k}$$Nun multiplizieren wir jede Spalte von $Q$ mit genau dieser Phase (im Code: Q * (dg / np.abs(dg))). Mathematisch entspricht das der Multiplikation mit einer unitären Diagonalmatrix $D$:$$B_j = Q \cdot D = Q \cdot \begin{pmatrix} \mathrm{e}^{i\phi_1} & & 0 \\ & \ddots & \\ 0 & & \mathrm{e}^{i\phi_N} \end{pmatrix}$$
Für die erste Spalte bewirkt das genau die nötige Korrektur und wir können leicht zeigen das das gefunden $B_j$ die Gleichung $B_j \vert{}0\dots0\rangle = \vert{}\chi_j\rangle$ tatsächlich erfüllt:

$$B_j \vert{}0\dots0\rangle = (B_j)_{:, 1} = Q_{:, 1} \cdot \mathrm{e}^{i\phi_1} = \big(\mathrm{e}^{-i\phi_1} \vert{}\chi_j\rangle\big) \cdot \mathrm{e}^{i\phi_1} = \vert{}\chi_j\rangle$$

Das Ergebnis: $B_j$ ist als Produkt zweier unitärer Matrizen ($Q$ und $D$) selbst garantiert unitär. Ihre erste Spalte ist nun fehlerfrei $\vert{}\chi_j\rangle$, womit $B_j \vert{}0\dots0\rangle = \vert{}\chi_j\rangle$ exakt erfüllt ist. Der Quantencomputer kann diese Matrix nun als fehlerfreies Gatter $B_j^\dagger$ kompilieren.

# Messergebnisse der Off-Diagonalterme in die richtige Basis transformieren

Bei Off-Diagonaltermen (Kohärenzen) $\rho_{ab}$ (mit $a \neq b$) stehen wir nun vor einem zusätzlichen Problem: Off-Diagonalterme sind komplexwertig ($\rho_{ab} \in \mathbb{C}$). Würden wir einfach die entsprechende Zeile $r_{ab}$ aus $R$ greifen und analog zu den Diagonaltermen das Betragsquadrat $q_{ab} = \vert{}\langle \chi_{ab} \vert{} y_t \rangle\vert{}^2$ messen, erhielten wir lediglich das Betragsquadrat $\vert{}\rho_{ab}\vert{}^2$. Alle Informationen über das komplexe Vorzeichen (die Phase) wären unwiederbringlich verloren, da $\rho_{ab}$ nicht zwingend reell und positiv ist.

Um das Problem zu lösen, transformieren wir den Zustand noch auf dem Quantencircuit in eine Basis, in der die komplexen Off-Diagonalterme als reine physikalische Populationen (die zwingend $\ge 0$ sind) in Erscheinung treten. 

Wir betrachten die Superpositionszustände $\vert{}+\rangle = \frac{1}{\sqrt{2}}(\vert{}a\rangle + \vert{}b\rangle)$ und $\vert{}R\rangle = \frac{1}{\sqrt{2}}(\vert{}a\rangle - i\vert{}b\rangle)$. Die physikalischen Populationen dieser Zustände sind:
$$\rho_{++} = \langle + \vert{} \rho_S \vert{} + \rangle = \frac{1}{2}(\rho_{aa} + \rho_{bb}) + \text{Re}(\rho_{ab})$$
$$\rho_{RR} = \langle R \vert{} \rho_S \vert{} R \rangle = \frac{1}{2}(\rho_{aa} + \rho_{bb}) + \text{Im}(\rho_{ab})$$

Da $\rho_{++}$ und $\rho_{RR}$ echte physikalische Besetzungswahrscheinlichkeiten sind, gilt zwingend $\rho_{++} \ge 0$ und $\rho_{RR} \ge 0$. Dadurch ist die Wurzelziehbarkeit wieder eindeutig gegeben und die unbekannte globale Phase von $y_t$ hebt sich im Betragsquadrat identisch weg!

Wir konstruieren nun die zugehörigen Vektor-Zeilen aus unserer Matrix $R$. Seien $r_{aa}, r_{bb}, r_{ab}$ und $r_{ba}$ die jeweiligen Zeilen aus $R$, die diese Elemente extrahieren. $r_{ab}$ ist also diejenige ganze Zeile der Matrix $R$, die bei Multiplikation mit $y_t$ das Element $\rho_{ab}$ liefert: $\rho_{ab} = r_{ab} \cdot y_t$. Diese Multiplikation gibt dann ja die vektorisierte Dichtematrix. Aufgrund der Linearität gilt für die neuen "Populations-Zeilen":
$$r_{++} = \frac{1}{2}(r_{aa} + r_{bb} + r_{ab} + r_{ba})$$
$$r_{RR} = \frac{1}{2}(r_{aa} + r_{bb} - i r_{ab} + i r_{ba})$$

Damit wir diese Vektoren in der Quantenwelt nutzen können, machen wir sie zu gültigen, auf Länge $1$ normierten Quantenzuständen: $\vert{}\chi_{++}\rangle := \frac{r_{++}^\dagger}{\Vert{}r_{++}\Vert{}_2}$ und $\vert{}\chi_{RR}\rangle := \frac{r_{RR}^\dagger}{\Vert{}r_{RR}\Vert{}_2}$. Wenn wir nun das quantenmechanische Skalarprodukt zwischen diesen neuen Zuständen und unserem Registerzustand $\vert{}y_t\rangle$ bilden, erhalten wir:
$$\langle \chi_{++} \vert{} y_t \rangle = \frac{r_{++}}{\Vert{}r_{++}\Vert{}_2} \cdot y_t = \frac{\rho_{++}}{\Vert{}r_{++}\Vert{}_2} \quad \implies \quad \rho_{++} = \Vert{}r_{++}\Vert{}_2 \cdot \langle \chi_{++} \vert{} y_t \rangle$$

Ein Quantencomputer kann jedoch nur messen wie viel eines Zustandes der Rechenbasis $\mathcal{B}_{\text{comp}} = \{\vert{}00\dots0\rangle, \vert{}00\dots1\rangle, \dots, \vert{}11\dots1\rangle\}$ im Endzustand des Circuits steckt. Also bauen wir uns für jede dieser neuen "Sites" einen Adapter: Ein unitäres Gatter $B_{++}$ (bzw. $B_{RR}$), das den Basis-Zustand $\vert{}0\dots0\rangle$ exakt auf unseren Wunschzustand $\vert{}\chi_{++}\rangle$ dreht:

$$ B_{++} \vert{}0\dots0\rangle = \vert{}\chi_{++}\rangle \qquad \implies \qquad \langle\chi_{++}|= \langle 0\dots0| B_{++}^\dagger $$

Gemessen wird dann das Betragsquadrat $q_{++} \equiv \big\vert{} \langle 0\dots0 \vert{} B_{++}^\dagger \vert{} y_t \rangle \big\vert{}^2$. Unter Hinzunahme der Skalierungen $\Vert{}\widetilde{\vec{\Sigma}}_0\Vert{}_2$ und $s^t$ sowie der Post-Selection-Erfolgswahrscheinlichkeit $\sqrt{p_{\text{total}}(t)}$ erhalten wir die reellen physikalischen Populationen:
$$\rho_{++}(t) = \underbrace{\Vert{}r_{++}\Vert{}_2 \cdot \sqrt{q_{++}(t)}}_{\text{aus der Basis-Rotation}} \;\times\; \underbrace{\Vert{}\widetilde{\vec{\Sigma}}_0\Vert{}_2 \cdot s^t \cdot \sqrt{p_{\text{total}}(t)}}_{\text{Skalierung \& Überlebensrate}}$$
$$\rho_{RR}(t) = \underbrace{\Vert{}r_{RR}\Vert{}_2 \cdot \sqrt{q_{RR}(t)}}_{\text{aus der Basis-Rotation}} \;\times\; \underbrace{\Vert{}\widetilde{\vec{\Sigma}}_0\Vert{}_2 \cdot s^t \cdot \sqrt{p_{\text{total}}(t)}}_{\text{Skalierung \& Überlebensrate}}$$

Zuletzt rekonstruieren wir den komplexen Off-Diagonalterm klassisch durch einfache Subtraktion der zuvor ermittelten Diagonalterme (Populationen) $\rho_{aa}$ und $\rho_{bb}$:
$$\text{Re}(\rho_{ab}(t)) = \rho_{++}(t) - \frac{1}{2}\big(\rho_{aa}(t) + \rho_{bb}(t)\big)$$
$$\text{Im}(\rho_{ab}(t)) = \rho_{RR}(t) - \frac{1}{2}\big(\rho_{aa}(t) + \rho_{bb}(t)\big)$$


### Die Fehlerfortpflanzung und Zahl der benötigten Shots
Das Auslesen der Off-Diagonalterme erfordert die Verrechnung von drei statistisch fehlerbehafteten Messgrößen ($\rho_{++}, \rho_{aa}, \rho_{bb}$). Da diese additiv verknüpft werden, addieren sich ihre absoluten Varianzen (Gaußsche Fehlerfortpflanzung). Der statistische Fehler des Realteils lautet:
$$\Delta \text{Re}(\rho_{ab}) \approx \sqrt{ (\Delta \rho_{++})^2 + \frac{1}{4}(\Delta \rho_{aa})^2 + \frac{1}{4}(\Delta \rho_{bb})^2 }$$

Die Anzahl der benötigten Treffer $N_{\text{Treffer}}$ muss daher bei Off-Diagonaltermen im Vorfeld höher angesetzt werden. Wir fordern für den relativen Fehler weiterhin:
$$\frac{\sigma}{\mu} = \frac{\Delta \text{Re}(\rho_{ab})}{\vert{}\text{Re}(\rho_{ab})\vert{}} \approx \frac{1}{\sqrt{N_{\text{Treffer}}}}$$
Unter der Annahme, dass die Varianzen aller Messungen ähnlich sind, verdreifacht sich die Gesamtvarianz durch die Subtraktion. Um die gleiche Zielgenauigkeit von $1\,\%$ zu erreichen, steigt die benötigte Shot-Zahl $N_{\text{shots, nötig}}$ für Kohärenzen grob um einen Faktor $2$ bis $3$ gegenüber reinen Diagonaltermen an:
$$N_{\text{shots, nötig}} \geq \frac{3}{4 \cdot 10^{-4} \cdot q_{\min} \cdot p_{t,\min}}$$

Um $q_{++}(t)$ und $p_{\text{total}}(t)$ vorab zu kennen, iteriert die Funktion den reduzierten Zustandsvektor klassisch durch die Zeit: $y_{t} = H_m \cdot y_{t-1}$. Die Überlebenswahrscheinlichkeit (Post-Selection) ist exakt der quadratische Normverlust relativ zur anvisierten Dilatations-Skalierung: $p_{\text{total}}(t) = \left( \frac{\Vert{}y_t\Vert{}}{s^t \Vert{}y_0\Vert{}} \right)^2$. Der Überlapp $q_{++}(t)$ ist das quantenmechanische Skalarprodukt des normierten Zustands mit unserem Zielzustand:
$$q_{++}(t) = \big\vert{} \langle \chi_{++} \vert{} \frac{y_t}{\Vert{}y_t\Vert{}} \rangle \big\vert{}^2$$


### Herleitung von $B_{++}$ (bzw. allgemein $B$)
Das unitäre Gatter $B$ muss die Bedingung $B |0\dots0\rangle = |\chi\rangle$ erfüllen. Da die Multiplikation mit dem Nullzustand $e_1 = (1, 0, \dots, 0)^\top$ die erste Spalte herausgreift, suchen wir eine unitäre Matrix mit $|\chi\rangle$ als erster Spalte:
$$B = \Big( |\chi\rangle \;\Big|\; v_2 \;\Big|\; \dots \;\Big|\; v_N \Big)$$

1. **Startmatrix:** Setze $|\chi\rangle$ in die erste Spalte der Einheitsmatrix $\mathbb{I}$:
   $$M = \Big( |\chi\rangle \;\Big|\; e_2 \;\Big|\; \dots \;\Big|\; e_N \Big)$$

2. **QR-Zerlegung:** Zerlege $M = Q \cdot R$. Da $R$ eine obere Dreiecksmatrix ist und $\||\chi\rangle\|_2 = 1$, gilt für die erste Spalte:
   $$M_{:, 1} = Q_{:, 1} \cdot R_{11} \implies Q_{:, 1} = \mathrm{e}^{-i\phi_1} |\chi\rangle \quad \text{mit } |R_{11}| = 1$$

3. **Phasen-Korrektur:** Multipliziere $Q$ mit der diagonalen Phasenmatrix $D = \operatorname{diag}(R_{kk}/|R_{kk}|)$:
   $$B = Q \cdot D \implies B |0\dots0\rangle = (B)_{:, 1} = Q_{:, 1} \cdot \mathrm{e}^{i\phi_1} = |\chi\rangle$$

$B$ ist garantiert unitär und implementiert die gewünschte Basisdrehung $B^\dagger$ für den Quantenschaltkreis.

# Die Auswirkung der Skalierung der ADOs und $\delta$ auf die Zahl der Shots und das conditioning

### Die Auswirkung von $\delta$ auf die Zahl der benötigten Shots: Ein Zielkonflikt

Die benötigte Anzahl an Shots hängt für eine geforderte relative Genauigkeit $\epsilon$ (z. B. $\epsilon = 0.01$ für $1\,\%$) zwingend vom Produkt aus der Projektionswahrscheinlichkeit $q_j(t)$ und der Überlebenswahrscheinlichkeit $p_{\text{total}}(t)$ ab:
$$N_{\text{shots}} \approx \frac{1}{4 \cdot \epsilon^2 \cdot q_j(t) \cdot p_{\text{total}}(t)}$$

Wie man am Graphen des Shot-Bedarfs erkennen kann, führt diese Abhängigkeit zu einer charakteristischen **Schüsselform (U-Kurve)**. Das bedeutet, dass weder ein extrem kleines noch ein extrem großes $\delta$ optimal ist. Der Grund hierfür ist, dass der Lyapunov-Shift $\delta$ auf die beiden Größen $p_{\text{total}}$ und $q_j$ stark *gegenläufig* wirkt:

**1. Der Post-Selection-Verlust bei großem $\delta$ (Die rechte Flanke)**
In der unitären Dilatation wird die Dämpfung $s = \mathrm{e}^{\delta \Delta t}$ dadurch realisiert, dass der unerwünschte Amplitudenanteil in den Ancilla-Fehlerzweig rotiert wird. Nach $t$ Zeitschritten ist die theoretische Wahrscheinlichkeit, dass das Ancilla-Qubit durchgehend $0$ misst (also der Shot nicht verworfen wird):
$$p_{\text{total}}(t) \propto s^{-2t} = \mathrm{e}^{-2\delta t \Delta t}$$
Mit wachsendem $\delta$ fällt $p_{\text{total}}(t)$ exponentiell steiler ab. Ein zu **großes $\delta$** vernichtet die Überlebensrate, wodurch die benötigte Shot-Zahl $N_{\text{shots}} \propto 1/p_{\text{total}}$ massiv ansteigt (rechter Ast der Kurve).

**2. Der Signalverlust durch Ableseverstärkung bei kleinem $\delta$ (Die linke Flanke)**
Dies ist der numerische Effekt, der das Signal auf dem Quantencomputer diktiert: Ein zu **kleines $\delta$** lässt das transiente Wachstum der unskalierten HEOM-Matrix fast ungebremst zu. Die Lyapunov-Metrik $W$ summiert dieses Wachstum auf. Dadurch wächst ihr größter Eigenwert extrem an ($\lambda_{\max} \propto 1/\delta$), während der kleinste Eigenwert in der Größenordnung von $1$ verbleibt ($\lambda_{\min} \approx 1$). Die Konditionszahl explodiert somit umgekehrt proportional zu $\delta$: $\operatorname{cond}(W) \propto 1/\delta$.

Das verzerrt die klassische *Ableseverstärkung* $A \propto \Vert r_j \Vert_2 \cdot \Vert y_0 \Vert_2$ massiv, da die beiden Vektornormen in entgegengesetzte Richtungen gezogen werden:
* **Die Startnorm $\Vert y_0 \Vert_2$:** Da $y_0 = W^{1/2} x_0$, wird ihre Länge durch den größten Eigenwert dominiert und skaliert mit $\sqrt{\lambda_{\max}} \propto 1/\sqrt{\delta}$.
* **Die Auslese-Norm $\Vert r_j \Vert_2$:** Da $r_j$ aus der inversen Matrix $R = \Pi W^{-1/2}$ stammt, skaliert sie mit dem Kehrwert des kleinsten Eigenwerts: $1/\sqrt{\lambda_{\min}} \approx 1$.

Die gesamte klassische Ableseverstärkung wächst bei kleinem $\delta$ also mit der Wurzel der Konditionszahl:
$$A \propto \sqrt{\operatorname{cond}(W)} \propto \frac{1}{\sqrt{\delta}}$$

Nun greift die physikalische Realität ein: Die gesuchte Systempopulation $\rho_{jj}(t)$ ist eine physikalische Konstante. Da die Gleichung $\rho_{jj} = A \cdot \sqrt{q_j} = \text{konstant}$ lautet, erzwingt ein gigantisches $A$ ein entsprechend winziges Signal $q_j$ auf der Quantenhardware:
$$q_j = \left( \frac{\rho_{jj}}{A} \right)^2 \propto \frac{1}{A^2} \propto \delta$$
Ein sehr kleines $\delta$ führt also zu einem astronomisch kleinen Signal $q_j \propto \delta$ im Quantenregister. Das Messrauschen überstrahlt das Signal völlig, und die benötigte Shot-Zahl $N_{\text{shots}} \propto 1/q_j$ explodiert in die Höhe (linker Ast der Kurve).

**3. Die mathematische Synthese: Die Schüsselform**
Setzt man die beiden gegenläufigen Abhängigkeiten ($q_j \propto \delta$ und $p_{\text{total}} \propto \mathrm{e}^{-2\delta t \Delta t}$) in die Fehlerfortpflanzung ein, erhält man die direkte Abhängigkeit der nötigen Shots von $\delta$:
$$N_{\text{shots}} \propto \frac{1}{q_j \cdot p_{\text{total}}} \propto \frac{1}{\delta \cdot \mathrm{e}^{-2\delta t \Delta t}}$$

Genau diese mathematische Funktion $f(\delta) = \frac{1}{\delta \cdot \exp(-c\delta)}$ erzeugt die charakteristische Schüsselform: 
Für $\delta \to 0$ explodiert der Term wegen des $1/\delta$ (Signalverlust), und für $\delta \to \infty$ explodiert er wegen der Exponentialfunktion (Post-Selection-Verlust). 

**Fazit: Das Minimum**
Wir stehen vor einem fundamentalen Zielkonflikt:
- **$\delta$ zu klein:** Die Metrik $W$ explodiert $\implies$ das Signal $q_j$ verschwindet $\implies N_{\text{shots}}$ wächst.
- **$\delta$ zu groß:** Die Post-Selection verwirft fast alles $\implies p_{\text{total}}$ verschwindet $\implies N_{\text{shots}}$ wächst.

Man wählt $\delta$ in der Praxis daher **nicht** so klein wie möglich, sondern sucht exakt das **Minimum dieser Schüsselkurve** (z. B. bei $\delta \approx 10^{-1}\,\text{cm}^{-1}$ im unskalierten Fall). An diesem "Sweet Spot" ist die Balance zwischen dem messbaren Überlapp $q_j$ und der Überlebensrate $p_{\text{total}}$ ideal, sodass sich die Populationen mit minimalem Shot-Bedarf ($N_{\text{shots}} \sim 10^4$) exakt rekonstruieren lassen.
### Die Auswirkung der Skalierung der ADOs auf die Zahl der benötigten shots
Wir haben zuvor gesehen, dass die Einträge der Hierarchie (die ADOs) über die Kopplungsterme stark asymmetrische Vorfaktoren erhalten. Der Term nach oben skaliert mit $\phi_j$, der Term nach unten mit $n_{jk} \theta_{j,k} \propto (n_{jk}+1) \frac{\vert{}a_{jk}\vert{}}{\hbar}$. Diese Asymmetrie macht den Liouville-Operator $\mathcal{L}_{\mathrm{HEOM}}$ extrem schlecht konditioniert ("steif"), was klassische Differentialgleichungslöser zwingt, mikroskopisch kleine Zeitschritte zu wählen.

Um dieses Problem klassisch zu beheben, führt man die skalierte HEOM (scale_ados=True) ein. Wir definieren einen Skalierungsfaktor, der exponentiell und faktoriell mit der Tiefe der Hierarchie wächst:$$S_{\mathbf{n}} = \sqrt{ \prod_{j,k} n_{jk}! \left( \frac{\vert{}a_{jk}\vert{}}{\hbar} \right)^{n_{jk}} } \quad \gg 1$$Und definieren unsere neuen, numerischen Variablen als:$$\tilde{\sigma}_{\mathbf{n}}(t) = S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}(t)$$

Setzt man $p_{\text{total}}(t) \propto \mathrm{e}^{-2\delta t \Delta t}$ in die Formel für die benötigten Shots ein, erhält man:$$N_{\text{shots}} \propto \frac{1}{p_{\text{total}}(t)} \propto \mathrm{e}^{+2\delta t \Delta t}$$

Betrachten wir den gesamten Zustandsvektor in unserer skalierten Basis:$$\widetilde{\vec{\Sigma}}(t) = \begin{pmatrix} \tilde{\sigma}_{\mathbf{0}}(t) \\ \tilde{\sigma}_{\mathbf{n}_1}(t) \\ \vdots \\ \tilde{\sigma}_{\mathbf{n}_{\max}}(t) \end{pmatrix} = \begin{pmatrix} S_{\mathbf{0}} \hat{\sigma}_{\mathbf{0}}(t) \\ S_{\mathbf{1}} \hat{\sigma}_{\mathbf{1}}(t) \\ \vdots \\ S_{\mathbf{n}_{\max}} \hat{\sigma}_{\mathbf{n}_{\max}}(t) \end{pmatrix}$$Da der Systemblock (die nullte Ordnung) keine Bad-Anregungen hat, ist sein Skalierungsfaktor $S_{\mathbf{0}} = 1$. Für alle tieferen ADOs wächst $S_{\mathbf{n}}$ jedoch gigantisch an (oft in die Größenordnung $10^3$ bis $10^6$).

In der physikalischen Realität ($\hat{\sigma}_{\mathbf{n}}$) sind die ADOs sehr kleine Korrekturen, weil das Bad nur schwach mit dem System korreliert ist (z.B. $\Vert\hat{\sigma}_{\mathbf{n}}\Vert \sim 10^{-4}$).In unserem skalierten, numerischen Vektor $\widetilde{\vec{\Sigma}}$ multiplizieren wir diese winzigen physikalischen Werte jedoch mit den gigantischen Faktoren $S_{\mathbf{n}}$!Das führt dazu, dass die numerischen ADO-Einträge plötzlich massiv anwachsen:$$\Vert \widetilde{\Sigma}_{\mathrm{ADOs}} \Vert \gg \Vert \widetilde{\Sigma}_{\mathrm{System}} \Vert$$

Der Quantencomputer repräsentiert unseren skalierten Zustandsvektor gezwungenermaßen als normierten Quantenzustand:$$\vert{}y_t\rangle = \frac{\widetilde{\vec{\Sigma}}(t)}{\Vert \widetilde{\vec{\Sigma}}(t) \Vert} = \begin{pmatrix} \text{Systemblock} \\ \text{ADOs} \end{pmatrix}$$Da die Norm des Nenners $\Vert \widetilde{\vec{\Sigma}}(t) \Vert$ wegen der Faktoren $S_{\mathbf{n}}$ riesig ist (z.B. $10^4$), wird unser Systemblock – der uns als Einziges interessiert und der nur mit $S_{\mathbf{0}}=1$ skaliert wurde – bei der Normierung unbarmherzig zusammengestaucht:$$\Vert y_{\text{System}}\Vert = \frac{\Vert\tilde{\sigma}_{\mathbf{0}}\Vert}{\Vert \widetilde{\vec{\Sigma}}(t) \Vert} \approx \frac{1}{10^4} = 10^{-4}$$Das System ist im Quantenregister nahezu "unsichtbar" geworden, da die ADOs $\approx 99{,}999999\,\%$ der Wahrscheinlichkeitsamplitude an sich gerissen haben. Die Wahrscheinlichkeit $q_j$, bei einer Messung unser System überhaupt im richtigen Zustand anzutreffen, schrumpft dadurch quadratisch:$$q_j = \Vert y_{\text{System}}\Vert^2 \approx (10^{-4})^2 = 10^{-8}$$
Wollen wir einen Fehler von $\sim 1\,\%$, können wir in die Gleichung für $N_{\text{shots}}$ von oben einsetzen :$$N_{\text{shots}} \approx \frac{1}{4 \cdot 10^{-4} \cdot q_j} \approx \frac{1}{10^{-4} \cdot 10^{-8}} = 10^{12} \text{ Shots}$$Um ein Signal von $q_j \sim 10^{-8}$ vom statistischen Rauschen zu unterscheiden, bräuchten wir eine Billion Shots. Daher müssen in der realtität immer die unskalierten ADOs verwendet werden.

### Die Auswirkung der Skalierung der ADOs auf die Operatornorm $||P||_2$
Wenn man die ADOs nicht skaliert würde die Norm von  $P(t) = \mathrm{e}^{\mathcal{L}t}$ exponetiell mit der Koplungsstärke $\lambda$ wachsen: $$\Vert P(t) \Vert_2 \approx \mathrm{e}^{\omega_{\max} t} \approx \mathrm{e}^{\frac{\lambda}{2} t}$$ wobei $\omega_\text{max}$ der größte Eigenwert von $\mathcal{L}$ ist. D.h. die Matrix $W$ muss das gesamte transiente Wachstum der unskalierten Matrix abfangen. Die Konditionszahl von $W$ skaliert quadratisch mit dem transienten Wachstum:$$\text{cond}(W) \sim \Vert P\Vert_2^2$$ Wenn die physikalische Kopplung $\lambda$ sehr stark ist, schießt $\Vert P\Vert_2$ bei scale_ados=False in die Millionen oder Milliarden. Die Konditionszahl $\text{cond}(W)$ übersteigt dann schnell den Wert $10^{16}$. Wenn $\text{cond}(W) > 10^{16}$ wird, geht bei der klassischen Berechnung von $W^{-1/2}$ (was wir für die Rekonstruktionsmatrix $R$ und das Gatter $U$ brauchen) die gesamte numerische Information im Rundungsrauschen verloren. Tatsächlich ist das Scheitern bei starker Kopplung in diesem Algorithmus kein fundamentales quantenmechanisches Limit, sondern primär ein reines Artefakt unserer klassischen 64-Bit-Standardarchitektur (Float64).  Wenn wir einen klassischen Computer (oder eine Software) nutzen würden, der mit beliebiger Genauigkeit (Arbitrary-Precision Arithmetic) rechnet, löst sich dieser Teufelskreis theoretisch auf. Die $\delta$-Schranke gilt natürlich noch immer. Es ist ja nur ein numerischer Fehler, kein theoretischer.

### Die Auswirkung des $\delta$ auf die Operatornorm $||P||_2$
Um die Dilatation $U$ auf dem Quantencomputer zu bauen, muss der Propagator eine strikte Kontraktion sein:$$\Vert{}\widetilde{P}\Vert{}_2 = \Vert{}W^{1/2} P_\delta W^{-1/2}\Vert{}_2 \le 1$$Die Metrik $W$ wird aus der zeitdiskreten Lyapunov-Gleichung bestimmt:$$P_\delta^\dagger W P_\delta - W = -Q \quad \text{mit } Q = \mathbb{I} - P_\delta^\dagger P_\delta > 0$$wobei $P_\delta = \mathrm{e}^{-\delta \Delta t} P = s^{-1} P$ der künstlich gedämpfte Propagator mit dem Dämpfungsfaktor $s = \mathrm{e}^{\delta \Delta t} > 1$ ist. Schreibt man die formale Lösung dieser Lyapunov-Gleichung als unendliche Neumann-Reihe auf:$$W = \sum_{k=0}^{\infty} \big(P_\delta^\dagger\big)^k Q \, P_\delta^k = \sum_{k=0}^{\infty} \mathrm{e}^{-2k\delta \Delta t} \big(P^\dagger\big)^k Q \, P^k$$

Ist $\delta$ klein, fällt der Dämpfungsterm $\mathrm{e}^{-2k\delta \Delta t}$ nur extrem langsam ab und die summanden und somit auch $||W||_2$ werden durch das Produkt der $P^k$ sehr groß. Die Folge ist, dass die Die Eigenwerte von $W$ extrem auseinander driften ($\lambda_{\max}(W) \sim 10^{14}$, $\lambda_{\min}(W) \sim 1$) und die Konditionszahl explodiert: $\operatorname{cond}(W) = \frac{\lambda_{\max}(W)}{\lambda_{\min}(W)} \sim 10^{14} \dots 10^{18}$. Das kann man sehen wenn man bedenkt, dass für eine beliebige Matrix die euklidische Operatornorm (Matrixnorm) $\Vert{}W\Vert{}_2$ über den größten Singulärwert definiert ist: $\Vert{}W\Vert{}_2 = \sigma_{\max}(W) = \sqrt{\lambda_{\max}(W^\dagger W)}$.

Ist $\delta$ jedoch groß, ist das Produkt $mathrm{e}^{-2k\delta \Delta t} \big(P^\dagger\big)^k$ schon nicht mehr so groß und daher wird der gesamte Summand nicht so groß und $W$ daher auch nicht. Weil die Summe nicht mehr so riesig wird, schrumpft $\lambda_{\max}(W)$ drastisch (z. B. von $10^{14}$ herunter auf $10^4$). Damit fällt $\operatorname{cond}(W)$ weit unter die kritische Float64-Grenze von $10^{16}$ in einen absolut sicheren Bereich (z. B. $10^4 \dots 10^6$).